# H3 LocalStudio — Colab mk32（環境快照＋FlashVSR／VOSR 背景下載）

**mk32（修正）：** 按「停止人臉精修」後，GPU 上的精修現在會真的被中斷；之前偶爾沒送出中斷，精修繼續在背景跑並卡住下一支影片。Studio 版本沒變，mk31 存的環境快照可直接沿用。

**mk31：對齊 LocalStudio repo `4cc3112`。**
- 圖生影片的首幀／尾幀裁切不再鎖定 16:9，可自由裁切；輸出影片依裁切後的首圖比例（沒有首圖時依尾圖）。這個修正已在 repo 內，不再靠補丁。
- 影片換角新增 **UntMods FaceSwap**（臉部身份替換，8 步專用 Turbo 或 20 步原生）。兩個 LoRA 首次使用時下載；**SHA256 尚未釘選、網址未在 Colab 實測**，第 1 格 `ENABLE_UNTMODS_FACESWAP=False` 可關閉。
- Studio 版本改變，舊的環境快照會自動失效：第一次執行走完整安裝並重存快照。


**mk30（修正）：** 用「QUALITY · 原生 12／16 步」做的影片可以「再做一次」了。之前後端存作品設定時只認得原版的設定名稱，把 Colab 專用的 `quality-12`／`quality-16` 丟掉，導致顯示「作品無原始設定」。


**mk29：FlashVSR、VOSR 模型改成背景下載，不再擋住開機。**
- 第 6 格只安裝節點、建立資料夾，模型（FlashVSR 約 6 GiB、VOSR 約 6.6 GiB）交給背景程式 `h3_bg_models` 在第 8 格起下載；一般生成可以立刻開始。
- 下載完成前就使用 FlashVSR 或 VOSR：工作會在 ComfyUI 佇列中**等下載完成再執行**，不會報錯（`h3_comfy.log` 會顯示「等待 … 背景下載」）。
- 下載失敗只影響該功能，錯誤寫在 `/content/h3_bg_models.log`；11b 每分鐘會顯示背景下載狀態。快速恢復格（R）會自動接續未完成的下載。
- 修臉偵測器（約 50 MB、1～2 秒）仍在第 6 格直接下載。
- 環境快照沿用 mk28 的格式，mk28 存的快照可以直接用。


**mk28：環境快照，加快開機。**
- 第一次完整安裝後，第 9b 格會在**背景**把 venv（含 Torch）、ComfyUI＋自訂節點、LocalStudio（含 `node_modules`）、uv 的 Python 3.11 打包成一個 `tar.zst`，存到 `MyDrive/H3_Colab/env_snapshots/`（約數 GB；不含模型與作品）。存檔期間可正常使用 Studio，看 `/content/h3_snapshot.log` 出現 `DONE` 才算完成，**完成前請勿刪除執行階段**。
- 之後的新執行階段：第 2 格直接從 Drive 解壓還原，跳過 uv／venv／Torch／pip／npm ci 安裝；第 4 格仍會實際在 GPU 上測 Torch 與 sol-attn，第 7 格仍會套補丁並重新 build（依你本次的設定）。
- 快照以版本 key 命名（各 repo commit、Torch 版本、CUDA wheel、Node 版本、修臉／FlashVSR／VOSR 開關）。任何一項改變就自動視為無效，改走完整安裝並存新快照、刪除舊快照。還原後快速檢查失敗也會自動退回完整安裝。
- 第 1 格 `USE_ENV_SNAPSHOT=False` 可完全停用；`REBUILD_ENV_SNAPSHOT=True` 強制完整安裝並重存（例如 Colab 映像大改版後怪怪的）。
- 模型仍沿用 mk26 的「用到才下載到 SSD」，不在快照裡。


**mk27（修正 mk26）：**
- 修正第 9 格修臉就緒檢查失敗（`missing: minimax_h3_ref2va_pruned_w4a8_mixed`）：H3-Studio 後端寫死 W4A8 檔名，mk25 因 Drive 兩種量化都在而通過；mk27 第 5 格對兩種 Ref2VA 量化都建立佔位檔，未選用的只有真的被載入時才下載。
- 修正第 9 格 `SINGULARITY_MODEL`／`SINGULARITY_LORA` 未定義（mk26 漏掉）。

以下為 mk26 說明：


使用 mk25 的固定 LocalStudio／ComfyUI 原始碼與全部生成功能；**唯一主要變更是模型取得方式**：

- **不再從 Google Drive 複製模型到 SSD**。H3、Qwen、SeedVR2、LoRA 等已知模型第一次被 ComfyUI 工作流載入時，會從 mk25 清單中既有的 Hugging Face／原作者下載網址**直接串流到 `/content/h3_models/`**；下載後進行 SHA256 驗證，再載入模型。相同 Colab 執行階段第二次使用就直接從 SSD 讀取。
- 第一次需要的模型仍要經歷**網路下載＋RAM／VRAM 載入兩個階段**。即時速度、進度、耗時會列在 `/content/h3_comfy.log`，也可使用 Studio 工作佇列的日誌提示。部分模型很大（例如 FL2VA 約 19.5 GiB）；請確保 Colab SSD 有足夠空間。
- 模型來源沿用 mk25 原清單（Comfy-Org、Kijai 與各模型原作者等 Hugging Face 倉庫），不是全部來自 MiniMax 官方帳號。下載 URL 和 SHA256 均沿用 mk25，未更換模型精度或生成演算法。
- 修臉偵測器、FlashVSR 和 VOSR 的可選模型因原先是安裝時準備的特別目錄，仍在**安裝階段**直接下載到 Colab SSD（不經 Drive）；可在第 1 格關閉對應 `INSTALL_...` 以減少啟動等待。
- Google Drive **繼續保存生成結果及設定**；不再存放／驗證必要的模型檔案。你自行放入 Drive 的自訂 LoRA 仍可直接讀取 Drive（沒有可信官方下載網址，所以不會假設能下載）。
- 原版 H3／Singularity V2、文生／首尾圖／Ref2VA、Turbo 4／6／8 步與 QUALITY 原生 12／16／20 步均依 mk25 保留；預設 Ref2VA INT8，不擅自改量化。
- **不需要執行原本「CPU 預先下載到 Drive」的第 0 格**。在 GPU Colab 從第 1 格按順序執行即可。你若按「全部執行」，第 0 格只會顯示提示，不下載。
- 若 Colab 重啟但 SSD 仍保留，可重跑第 1～11 格；若「中斷連線並刪除執行階段」，SSD 快取也會消失，下次需要重新下載。

注意：零位元組模型清單佔位檔僅供 ComfyUI 顯示可選項，**不是真正的模型**。生成時由 mk26 下載節點攔截並驗證完成才交給載入器；如果網路或 SSD 空間不足，會明確失敗，不會退回從 Drive 複製。此次修改已做靜態與模擬測試，但仍需真實 Colab GPU 測試各模式第一次載入。

參考：[Google Colab 使用限制](https://research.google.com/colaboratory/faq.html)。如透過網頁生成，請遵守 Colab 適用方案與運算單元限制。


In [ ]:
# 0. mk32 不需要 CPU 預下載：直接跳過本格。
# 模型在 GPU Colab 中「第一次被工作流載入時」才從其原始 Hugging Face／作者 URL 下載到 SSD。
print('mk31：不需要 CPU 預下載到 Google Drive。請從第 1 格開始（或按全部執行）。')


In [ ]:
# 1. 掛載 Google Drive（只用於輸出備份及自訂 LoRA），官方模型直接下載到 Colab SSD。
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json, getpass

drive.mount('/content/drive')
ROOT=Path('/content')
DRIVE_ROOT=ROOT/'drive/MyDrive/H3_Colab'
DRIVE_MODELS_ROOT=DRIVE_ROOT/'models'    # 僅供使用者自行提供的 LoRA；不作為官方模型來源
MODELS_ROOT=ROOT/'h3_models'            # mk26：官方模型全部直接下載到本地 SSD
DRIVE_OUTPUTS_ROOT=DRIVE_ROOT/'outputs'
OUTPUTS_ROOT=ROOT/'h3_outputs'
REPO=ROOT/'LocalStudio'; COMFY=ROOT/'ComfyUI'; WEB=REPO/'h3-local-studio'
VENV=ROOT/'h3_venv'; PYTHON=VENV/'bin/python'
# mk26：首次載入時從來源網址直接下載到 SSD；不再從 Drive copy。
LAZY_MODEL_DOWNLOAD=True
# 官方模型下載均固定做 SHA256 完整驗證；已驗證的模型在同一執行階段直接重用。
VERIFY_MODEL_SHA256=True
# 保留多少 SSD 空間，避免模型下載填滿整個 Colab 根磁碟。
MODEL_DOWNLOAD_RESERVE_GIB=8
# 這些模型在首次使用時下載；設為 False 會從 UI 隱藏相關選項。
ENABLE_QWEN=True
ENABLE_NOCTQ=True
ENABLE_CAMERA=True
ENABLE_SWAP=True
ENABLE_SEEDVR2=True
ENABLE_ORB360=True
# UntMods FaceSwap（影片換角的臉部替換模式）＋專用 8 步 Turbo LoRA；需要 ENABLE_SWAP=True。
ENABLE_UNTMODS_FACESWAP=True
# 沒有公開直連 URL 的自訂 LoRA，可繼續直接讀 Drive（不會複製到 SSD）。
ENABLE_DRIVE_CUSTOM_LORAS=True
INSTALL_FACE_REFINE=True
# Ref2VA 模型量化：首次用到時從原作者／Comfy-Org URL 下載到 Colab SSD。
#   'auto'：VRAM ≥ 40GB 選 int8，其餘選 w4a8（不依 Drive 檔案決定）。
#   'int8'：minimax_h3_ref2va_pruned_int8_convrot（約 19.5 GiB，畫質較好、吃 VRAM；首次使用時直接下載到 SSD）
#   'w4a8'：minimax_h3_ref2va_pruned_w4a8_mixed（約 11 GiB，L4 較不易 offload）
# 改這個值之後要重跑第 5、7 格（第 7 格會重新 build），快速恢復格不會套用新設定。
REF2VA_QUANT='int8'
assert REF2VA_QUANT in ('auto','int8','w4a8'), "REF2VA_QUANT 只能是 'auto'、'int8'、'w4a8'"
# Optional FlashVSR (install 時從作者倉庫直接下載至 Colab SSD)。
INSTALL_FLASHVSR=True
# Singularity V2 主模型＋專用加速 LoRA（首次用到時從來源下載到 SSD）。
INSTALL_SINGULARITY=True
# VOSR 1× 精修（安裝時從作者倉庫下載到 Colab SSD）。
INSTALL_VOSR=True
# mk28：環境快照（存在 MyDrive/H3_Colab/env_snapshots；只含程式與套件，不含模型與作品）。
USE_ENV_SNAPSHOT=True        # 有可用快照就還原，沒有就完整安裝後在背景存一份
REBUILD_ENV_SNAPSHOT=False   # True：忽略舊快照，完整安裝並重存
BASIC_USER='h3'
while True:
    BASIC_PASSWORD=getpass.getpass('設定 Studio 密碼（至少 8 字元，不存進 Drive；登入帳號是 h3）：')
    if len(BASIC_PASSWORD)>=8: break
    print(f'⚠ 密碼只有 {len(BASIC_PASSWORD)} 個字元，至少要 8 個，請重新輸入。',flush=True)
GEMINI_KEY=getpass.getpass('Gemini TTS API key（選填，直接 Enter 略過）：').strip()
for folder in (MODELS_ROOT,DRIVE_OUTPUTS_ROOT,OUTPUTS_ROOT): folder.mkdir(parents=True,exist_ok=True)
print('模型根目錄（Colab SSD，直接網路下載）:',MODELS_ROOT)
print('生成位置:',OUTPUTS_ROOT,'→ 完成後複製到',DRIVE_OUTPUTS_ROOT)


In [ ]:
# 2. 系統工具與獨立 Python 環境，不改動 Colab kernel 的 Torch。mk28：有可用環境快照時直接從 Drive 還原。
import shutil, signal, time, urllib.request, hashlib, re

# mk28：所有影響快照內容的版本集中在這裡（第 3／4／6 格引用）。
STUDIO_COMMIT='4cc3112bfdf75a40efc86e45cd5532a43c96ee9d'
COMFY_COMMIT='73c9bad4d21e7addbe1d13bc92eee0f1431b017d'
CUSTOM_NODE_COMMITS=[
 ('Larryvrh/ComfyUI-MiniMax-H3-Turbo','ComfyUI-MiniMax-H3-Turbo','55fee864dd7b2976b1c4ce3c3d5f7968f181409f'),
 ('NikoDemon80/ComfyUI-H3-Motion-Context','ComfyUI-H3-Motion-Context','c140ae99b8c38f782ebd8564c267b42aacade6a4'),
 ('vizart-vj/ComfyUI-MiniMax-H3-LongMedia','ComfyUI-MiniMax-H3-LongMedia','96c50f7febc8c2e80c2a50d030d6cb3ff3de7ee3'),
 ('kijai/ComfyUI-KJNodes','ComfyUI-KJNodes','073efb07419f56cc714e099a82e49fbc23ad9263')]
FACE_REFINE_COMMIT='d8521d14fe0d721d80cd9417fff5a559cbc21aba'
FLASHVSR_COMMIT='4820b3f02347bddcbbb9a5a85ab7638fe976366e'
VOSR_COMMIT='6a21810b3f4f1a0ffc477c55330db0ac03c671f0'
TORCH_PKGS=['torch==2.10.0','torchvision==0.25.0','torchaudio==2.10.0']
# 修改第 4／6 格的 pip 套件清單時，請一併改這個 schema 字串，讓舊快照失效。
SNAPSHOT_SCHEMA='mk28-env-1'

def run(args,**kwargs):
    subprocess.run([str(x) for x in args],check=True,**kwargs)

def process_token(pid):
    try:
        fields=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
        return None if fields[0]=='Z' else fields[19]
    except (FileNotFoundError,ProcessLookupError):
        return None

def stop_managed(name):
    record=ROOT/(name+'.pid.json')
    if not record.exists(): return
    saved=json.loads(record.read_text()); pid=saved['pid']
    if process_token(pid)==saved['token']:
        os.killpg(pid,signal.SIGTERM)
        for _ in range(50):
            if process_token(pid)!=saved['token']: break
            time.sleep(0.1)
        if process_token(pid)==saved['token']: os.killpg(pid,signal.SIGKILL)
    record.unlink(missing_ok=True)

# Only stop processes previously launched and identified by this notebook.
for name in ('h3_bg_models','h3_snapshot','h3_cloudflared','h3_drive_copy','h3_studio','h3_comfy'): stop_managed(name)
run(['apt-get','update','-qq'])
run(['apt-get','install','-y','-qq','ffmpeg','git','curl','nginx','apache2-utils','ca-certificates','build-essential','ninja-build','python3-venv','libgl1','libglib2.0-0','zstd'])
major=int(subprocess.check_output(['node','--version'],text=True).strip().lstrip('v').split('.')[0]) if shutil.which('node') else 0
if major!=24:
    run(['bash','-o','pipefail','-c','curl -fsSL https://deb.nodesource.com/setup_24.x | bash -'])
    run(['apt-get','install','-y','-qq','nodejs'])
# ---- mk28：環境快照還原 ----
try:
    _cuda=re.search(r'release (\d+)\.(\d+)',subprocess.check_output(['nvcc','--version'],text=True))
    TORCH_INDEX='cu130' if _cuda and int(_cuda.group(1))>=13 else 'cu128'
except (OSError,subprocess.CalledProcessError):
    TORCH_INDEX='unknown'
SNAPSHOT_KEY=hashlib.sha256(json.dumps({
    'schema':SNAPSHOT_SCHEMA,'studio':STUDIO_COMMIT,'comfy':COMFY_COMMIT,'nodes':CUSTOM_NODE_COMMITS,
    'face':FACE_REFINE_COMMIT if INSTALL_FACE_REFINE else None,
    'flashvsr':FLASHVSR_COMMIT if INSTALL_FLASHVSR else None,
    'vosr':VOSR_COMMIT if INSTALL_VOSR else None,
    'torch':TORCH_PKGS,'index':TORCH_INDEX,'node':24,'python':'3.11'},sort_keys=True).encode()).hexdigest()[:16]
SNAP_DIR=DRIVE_ROOT/'env_snapshots'
SNAP_FILE=SNAP_DIR/f'h3_env_{SNAPSHOT_KEY}.tar.zst'
SNAP_OK=SNAP_FILE.with_name(SNAP_FILE.name+'.ok')
ENV_FROM_SNAPSHOT=False

def snapshot_valid():
    try:
        return SNAP_FILE.is_file() and SNAP_OK.is_file() and int(SNAP_OK.read_text().split()[0])==SNAP_FILE.stat().st_size
    except (OSError,ValueError,IndexError):
        return False

def wipe_env():
    for p in (VENV,COMFY,REPO):
        if p.is_symlink() or p.exists():
            assert p.resolve().is_relative_to(ROOT) and not p.resolve().is_relative_to(DRIVE_ROOT)
            shutil.rmtree(p)

if USE_ENV_SNAPSHOT and not REBUILD_ENV_SNAPSHOT and not PYTHON.exists() and snapshot_valid():
    print(f'找到環境快照 {SNAP_FILE.name}（{SNAP_FILE.stat().st_size/1024**3:.2f} GiB），從 Drive 還原…',flush=True)
    t0=time.monotonic()
    try:
        wipe_env()
        run(['tar','-I','zstd -T0','-xf',SNAP_FILE,'-C','/'])
        run([PYTHON,'-c','import torch, comfy_kitchen, psutil; print("torch", torch.__version__)'])
        assert (WEB/'node_modules').is_dir() and (COMFY/'main.py').is_file() and (REPO/'.git').is_dir(), '快照內容不完整'
        ENV_FROM_SNAPSHOT=True
        print(f'✅ 環境已從快照還原（{time.monotonic()-t0:.0f} 秒），略過 uv／venv／Torch／pip／npm 安裝。',flush=True)
    except Exception as exc:
        print('⚠ 快照還原失敗，改為完整安裝：',type(exc).__name__,str(exc)[:300],flush=True)
        wipe_env()
elif USE_ENV_SNAPSHOT and REBUILD_ENV_SNAPSHOT:
    print('REBUILD_ENV_SNAPSHOT=True：完整安裝，完成後第 9b 格會重存快照。',flush=True)
elif USE_ENV_SNAPSHOT and not PYTHON.exists():
    print(f'沒有可用快照（key {SNAPSHOT_KEY}），本次完整安裝；完成後第 9b 格會在背景存快照。',flush=True)

if not ENV_FROM_SNAPSHOT:
    # Colab system Python can be 3.13 and python3-venv/ensurepip may not match it.
    # Install isolated Python 3.11 via uv, then seed pip in the venv.
    # This avoids modifying Colab kernel Torch and works across Python 3.12/3.13 images.
    run([sys.executable,'-m','pip','install','-q','uv'])
    UV=[sys.executable,'-m','uv']
    run([*UV,'python','install','3.11'])
    if PYTHON.exists():
        valid=subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True,text=True)
        version=subprocess.run([str(PYTHON),'-c','import sys; print(sys.version_info[:2])'],capture_output=True,text=True)
        if valid.returncode!=0 or version.returncode!=0 or '(3, 11)' not in version.stdout:
            print('Removing incomplete or incompatible venv:',VENV,flush=True)
            assert VENV.resolve().is_relative_to(ROOT)
            shutil.rmtree(VENV)
    if not PYTHON.exists():
        run([*UV,'venv','--python','3.11','--seed',str(VENV)])
    run([PYTHON,'-m','pip','install','--upgrade','pip','setuptools','wheel','packaging','ninja'])
print('venv Python:',subprocess.check_output([str(PYTHON),'--version'],text=True).strip(),flush=True)
cloud_url='https://github.com/cloudflare/cloudflared/releases/download/2026.10.3/cloudflared-linux-amd64'
cloud=Path('/usr/local/bin/cloudflared')
expected='053b80b0a23031a038fdbf3c073f59f82b8f1f152aa67971d7383ffafc0dbab0'
if not cloud.exists() or hashlib.sha256(cloud.read_bytes()).hexdigest()!=expected:
    run(['curl','--fail','--location','--retry','3',cloud_url,'-o',cloud])
assert hashlib.sha256(cloud.read_bytes()).hexdigest()==expected, 'cloudflared SHA256 mismatch'
cloud.chmod(0o755)
print('獨立 Python:',PYTHON)


In [ ]:
# 3. 固定原始碼版本；重新執行會回復 Colab checkout 後再套用下一格的補丁。
# 僅重置這份 notebook 所管理的 /content 目錄。
# mk28：STUDIO_COMMIT／COMFY_COMMIT／CUSTOM_NODE_COMMITS 定義在第 2 格（快照 key 也用它們）。
def checkout(url,path,commit,patch_files=()):
    path=Path(path)
    assert path.resolve().is_relative_to(ROOT) and not path.resolve().is_relative_to(DRIVE_ROOT)
    if not (path/'.git').exists(): run(['git','clone',url,path])
    actual=subprocess.check_output(['git','-C',path,'remote','get-url','origin'],text=True).strip()
    assert actual.rstrip('/').removesuffix('.git')==url.rstrip('/').removesuffix('.git'), f'Unexpected repo at {path}'
    # mk28：快照還原的 repo 已有該 commit 時不必再 fetch。
    if subprocess.run(['git','-C',path,'cat-file','-e',commit+'^{commit}'],capture_output=True).returncode:
        run(['git','-C',path,'fetch','origin',commit])
    present=[f for f in patch_files if (path/f).exists()]
    if present: run(['git','-C',path,'restore','--',*present])
    run(['git','-C',path,'checkout','--detach',commit])
    assert subprocess.check_output(['git','-C',path,'rev-parse','HEAD'],text=True).strip()==commit

checkout('https://github.com/shellpig/LocalStudio.git',REPO,STUDIO_COMMIT,
         ['h3-local-studio/app/page.tsx','h3-local-studio/lib/comfy.ts','h3-local-studio/lib/tts.ts','h3-local-studio/lib/drama3.ts','h3-local-studio/app/speech-studio.tsx','h3-local-studio/app/swap-studio.tsx','integrations/ComfyUI-H3-Studio/h3_studio_api.py','integrations/ComfyUI-H3-Studio/h3_studio_enhance.py','h3-local-studio/lib/enhance.ts','h3-local-studio/app/enhance-studio.tsx','h3-local-studio/lib/face-refine.ts','h3-local-studio/app/generation-queue-panel.tsx','h3-local-studio/lib/vosr.ts','integrations/ComfyUI-H3-Studio/h3_studio_vosr.py','integrations/ComfyUI-H3-Studio/h3_studio_swap.py','h3-local-studio/lib/swap.ts'])
checkout('https://github.com/comfyanonymous/ComfyUI.git',COMFY,COMFY_COMMIT)
for url,name,commit in CUSTOM_NODE_COMMITS:
    checkout('https://github.com/'+url+'.git',COMFY/'custom_nodes'/name,commit)
link=COMFY/'custom_nodes/ComfyUI-H3-Studio'
if link.is_symlink():
    assert link.resolve()==(REPO/'integrations/ComfyUI-H3-Studio').resolve()
elif link.exists():
    raise RuntimeError(f'{link} already exists and is not the expected symlink')
else:
    link.symlink_to(REPO/'integrations/ComfyUI-H3-Studio',target_is_directory=True)


In [ ]:
# 4. Linux Torch + Comfy Kitchen sol-attn 檢查（mk24：不再安裝 SageAttention）。失敗會停止，不會假裝已啟用加速。
import re
assert shutil.which('nvidia-smi'), '請先選擇 GPU 執行階段'
run(['nvidia-smi'])
assert shutil.which('nvcc'), '此執行階段缺少 CUDA 工具（nvcc），無法判斷 Torch／CUDA 版本'
nvcc=subprocess.check_output(['nvcc','--version'],text=True)
m=re.search(r'release (\d+)\.(\d+)',nvcc)
assert m, nvcc
cuda=tuple(map(int,m.groups()))
assert cuda>=(12,8), '需要 CUDA toolkit 12.8 或更新的 Colab 執行階段'
index='cu130' if cuda[0]>=13 else 'cu128'
if ENV_FROM_SNAPSHOT:
    assert index==TORCH_INDEX, f'快照的 Torch wheel 是 {TORCH_INDEX}，本機 CUDA 需要 {index}；請設 REBUILD_ENV_SNAPSHOT=True 後從第 1 格重跑'
    print('mk28：Torch／ComfyUI 依賴來自環境快照，略過 pip 安裝；以下仍在 GPU 上實測。',flush=True)
else:
    run([PYTHON,'-m','pip','install',*TORCH_PKGS,'--index-url','https://download.pytorch.org/whl/'+index])
    run([PYTHON,'-m','pip','install','-r',COMFY/'requirements.txt'])
    run([PYTHON,'-m','pip','install','-r',COMFY/'custom_nodes/ComfyUI-KJNodes/requirements.txt'])
probe="""
import torch,psutil
assert torch.cuda.is_available(), 'Torch CUDA 未就緒；檢查 Colab 驅動與 wheel 是否相容'
p=torch.cuda.get_device_properties(0)
assert torch.cuda.get_device_capability(0) in [(8,0),(8,6),(8,9),(9,0)], '此 notebook 支援 A100/Ampere、L4/Ada、H100/Hopper，請更換 GPU'
assert p.total_memory>=11.5*1024**3, '需要至少 12GB VRAM'
print(p.name, round(p.total_memory/1024**3,1),'GiB VRAM; RAM',round(psutil.virtual_memory().total/1024**3,1),'GiB')
if psutil.virtual_memory().total<28*1024**3: print('注意：低於 28GiB RAM，請優先改用高 RAM 執行階段以減少換頁')
"""
run([PYTHON,'-c',probe])
# mk24: sol-attn only. SageAttention is no longer installed; verify the Comfy Kitchen kernels by running them on the GPU.
probe="""
import torch,comfy_kitchen as ck
q=torch.randn(1,2,512,128,device='cuda',dtype=torch.bfloat16)
assert ck.sol_attn_is_available(q.device), 'Comfy Kitchen sol-attn kernel unavailable'
assert ck.int8_attention_is_available(q.device), 'Comfy Kitchen INT8 attention unavailable'
z=ck.sol_attn(q.transpose(1,2).contiguous(),q.transpose(1,2).contiguous(),q.transpose(1,2).contiguous(),tau=1.3)
torch.cuda.synchronize(); assert torch.isfinite(z).all()
print('Kitchen sol-attn GPU checks passed')
"""
run([PYTHON,'-c',probe])
if not ENV_FROM_SNAPSHOT: run(['npm','ci'],cwd=WEB)  # mk28：快照已含 node_modules


## mk26 模型來源與 SSD 下載

模型清單與 SHA256 沿用 mk25，預設完全不需要在 `MyDrive/H3_Colab/models` 放模型。第 5 格只建立**零位元組佔位檔**讓 ComfyUI 顯示模型名稱；第一次真正生成時才從原 Hugging Face／作者網址直接下載到 `/content/h3_models/<分類>/`，驗證 SHA256 後載入。下載時 `/content/h3_comfy.log` 會報告 GiB、%、MiB/s 與總秒數。

Drive 仍會備份 `/content/h3_outputs` 中完成的作品到 `MyDrive/H3_Colab/outputs`，用戶自行放的其他 LoRA 可從 Drive 直接讀取。**切勿在模型下載中途停止 ComfyUI 或刪除 SSD 快取**；下載中斷的 `.part` 可續傳。FlashVSR／VOSR 在第 6 格安裝時直接下載到 SSD（若要更快啟動，可在第 1 格先關閉）。


In [ ]:
# 5. mk32（同 mk26）：模型清單顯示在 ComfyUI；用到時才從 Hugging Face／作者網址下載至 Colab SSD。
# 不再掃描 Drive 模型、不會複製 Drive 模型；生成時自動續傳、SHA256 驗證及顯示速度。
import json, os, shutil, time, subprocess, logging
from pathlib import Path

# 下列 URL 和 SHA256 均取自 mk25 原始模型目錄；不同模型使用 Comfy-Org／作者各自發布來源。
CATALOG={'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors': {'folder': 'text_encoders', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/e5eb578a89295337b8ff433a035929ce0279e0b6/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'sha256': '35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6'}, 'minimax_h3_video_vae_int8_convrot.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/d8023be02fefbb3633b0cd335c3879f91177299d/minimax_h3_video_vae_int8_convrot.safetensors', 'sha256': '9bb2d96f218c76babd85e0611b85ca8fb330a90546c01a0005e8a58a59593410'}, 'minimax_h3_audio_vae_fp32.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/e5eb578a89295337b8ff433a035929ce0279e0b6/vae/minimax_h3_audio_vae_fp32.safetensors', 'sha256': '8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48'}, 'minimax_h3_fl2va_pruned_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/e5eb578a89295337b8ff433a035929ce0279e0b6/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors', 'sha256': 'e889202c41dafb67b10d67b97f0d8541508036a6090af23425a5c2615d03c47a'}, 'minimax_h3_ref2va_pruned_w4a8_mixed.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/d8023be02fefbb3633b0cd335c3879f91177299d/minimax_h3_ref2va_pruned_w4a8_mixed.safetensors', 'sha256': 'de2c6c29c4ee702b45e48e40daae3834aeee58ab681c732d9152589a87c89910'}, 'minimax_h3_ref2va_pruned_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors?download=true', 'sha256': '9255f52b6677845ad238f20dfaafa94727053694127ab7f255c048f0f9365779'}, 'minimax_h3_video_vae_fp16.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/e5eb578a89295337b8ff433a035929ce0279e0b6/vae/minimax_h3_video_vae_fp16.safetensors', 'sha256': '7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522'}, 'minimax_h3_turbo_v4_step600_ema.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/larryvrh/MiniMax-H3-Turbo-Lora/resolve/43a74557ac3f6539db8e0f2a959d03feb7a81480/minimax_h3_turbo_v4_step600_ema.safetensors', 'sha256': '5f3a626cd72c93a8b9318d6760c510bc5092d2ab13aaba1f932c5bab07a416d3'}, 'qwen_image_2.1_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/df94239739eef7205973a145ffa6f441f7b64e84/diffusion_models/qwen_image_2.1_int8_convrot.safetensors', 'sha256': 'cb74113cb03faecd79611b01fd7fd642f0aa60d6f0b95086abee214d75eaa57d'}, 'qwen3vl_8b_int8_convrot.safetensors': {'folder': 'text_encoders', 'url': 'https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/df94239739eef7205973a145ffa6f441f7b64e84/text_encoders/qwen3vl_8b_int8_convrot.safetensors', 'sha256': '8bfd0f6e12abf2d2d697ecc888e5e90b0d6741d6708f05799f53afa560452e8f'}, 'qwen_image_2.1_vae_bf16.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/df94239739eef7205973a145ffa6f441f7b64e84/vae/qwen_image_2.1_vae_bf16.safetensors', 'sha256': 'bb21f7473051e1ac368515dd3f2e15cd44d7a11748ee8823e1ddca3e4876b7c9'}, 'qwen_image_2.1_multiple_angles_v2_step1500.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/akhaliq/Qwen-Image-2.1-Multiple-Angles-LoRA/resolve/0d91c4dfa22b3f5aa89c020eadc30d8f42cb51c0/checkpoints_v2/angles_v2_full_qwen21_multiple_angles_v2_qwen21_multiple_angles_v2_000001500.safetensors', 'sha256': '548c57977c3f1c09c2c9e5f4e83e930cbc6f719c878a576ab8666ce29fef39b6'}, 'minimax_h3_head_swap_v1.0_r32.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/e095edae3f4c69e18b4f27ed9e541078c87a8d98/loras/minimax_h3_head_swap_v1.0_r32.safetensors', 'sha256': 'ea97bd1a8178460661055784d63b4cb560732196fd15d5d7f33bfd214b65ab00'}, 'minimax_h3_character_swap_v1.0_ref2va_r64_gl.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/Alissonerdx/Minimax-H3-ComfyUI/resolve/e095edae3f4c69e18b4f27ed9e541078c87a8d98/loras/minimax_h3_character_swap_v1.0_ref2va_r64_gl.safetensors', 'sha256': '643a3fe63646fee5aebb181a7e1c9f8a1feb5f472709f37e22eeec290cbb75e3'}, 'h3_character_swap_v1.5.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/akatz-ai/MiniMax-H3-Character-Swap-LoRA/resolve/96ea3407b230b6f7df71c58c70e31df96bd7e533/h3_character_swap_v1.5.safetensors', 'sha256': '8f2c3c3eba161df8d0c6099598b4730f641f5ce01e04cd16d2427cff8a35218f'}, 'seedvr2_3b_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/diffusion_models/seedvr2_3b_int8_convrot.safetensors', 'sha256': 'c3dec8bcc5916843a8a858572970597462e1f2dc598d6dfd818f6cd40f53a157'}, 'seedvr2_ema_vae_fp16.safetensors': {'folder': 'vae', 'url': 'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/vae/seedvr2_ema_vae_fp16.safetensors', 'sha256': '20678548f420d98d26f11442d3528f8b8c94e57ee046ef93dbb7633da8612ca1'}, 'NoctQ_V4_int8_convrot.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/Noctaluna/Noct-Q-Uncensored-Qwen-Image-2.1/resolve/main/NoctQ_V4_int8_convrot.safetensors', 'sha256': '4d92d5538253ab36e6a73b056cce5f697950f4303cb9e6686a3198d0ea13f9e8'}, 'REDQW21_V2_int8.safetensors': {'folder': 'diffusion_models', 'url': 'https://civitai.com/api/download/models/3370753?fileId=3258922', 'sha256': '30532b445eabd4b84b108168b6b3fe720535ef400f6bf9d68bd551817d3531a3'}, 'minimax_h3_orb360_step1500.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/MATLOWAI/MiniMax-H3-ORB360-CardSpin/resolve/de8aa4db6f9a0d949855f3dc62948ffc691aa6ce/minimax_h3_orb360_step1500.safetensors', 'sha256': 'e9275b2f456f6bd004c9cb3cf5c636cad87217c5bc96e6a5b5533e917de121da'}, 'Minimax-h3_Singularity_ref2va_v2_pruned_int8.safetensors': {'folder': 'diffusion_models', 'url': 'https://huggingface.co/WarmBloodAban/Minimax-h3_Singularity/resolve/1259cf16cd0b655ba691a32fb46a409142762972/Minimax-h3_Singularity_ref2va_v2_pruned_int8.safetensors', 'sha256': 'd7637c2c35cd30facd9e93778c491f6d82f91c4a441aecbdb4c384a62a5f3d9f', 'gib': 19.6}, 'minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors': {'folder': 'loras', 'url': 'https://huggingface.co/lightx2v/Minimax-h3-Turbo/resolve/30f485ed874aa44c8183348ad361c5eabad318da/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors', 'sha256': '5b9ab5ade15d0775676d01a907268a69a1468dc6033b3b0d3ded5502f3ebb84c', 'gib': 1.9}}
# mk31：UntMods FaceSwap（repo 4cc3112 新增的換角模式）。這兩個 LoRA 的 SHA256 尚未釘選（None＝不驗證雜湊，
# 首次下載完成後 h3_comfy.log 會印出實際 SHA256，可填回這裡）；網址用 main 分支，尚未在 Colab 實測。
CATALOG['SS_FaceSwap_MiniMax_H3_REF2VA.safetensors']={'folder':'loras','url':'https://huggingface.co/UntMods/FaceSwap_MiniMaxH3_REF2VA/resolve/main/SS_FaceSwap_MiniMax_H3_REF2VA.safetensors','sha256':None}
CATALOG['minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors']={'folder':'loras','url':'https://huggingface.co/lightx2v/Minimax-h3-Turbo/resolve/main/minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors','sha256':None}
REF2VA_FILES={'int8':'minimax_h3_ref2va_pruned_int8_convrot.safetensors',
              'w4a8':'minimax_h3_ref2va_pruned_w4a8_mixed.safetensors'}
try:
    GPU_VRAM_GIB=int(subprocess.check_output(['nvidia-smi','--query-gpu=memory.total','--format=csv,noheader,nounits'],text=True).split()[0])/1024
except Exception:
    GPU_VRAM_GIB=0
_quant=globals().get('REF2VA_QUANT','auto')
if _quant=='auto': _quant='int8' if GPU_VRAM_GIB>=38 else 'w4a8'
REF2VA_SELECTED=_quant
REF2VA_FILE=REF2VA_FILES[_quant]
if _quant=='int8' and 0<GPU_VRAM_GIB<38:
    print(f'注意：GPU 只有 {GPU_VRAM_GIB:.0f} GiB VRAM，Ref2VA INT8 可能需要 RAM offload。',flush=True)
print(f'Ref2VA 模型：{REF2VA_FILE}；首次載入時下載到 SSD。',flush=True)
COMMON=['qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
        'minimax_h3_video_vae_int8_convrot.safetensors',
        'minimax_h3_audio_vae_fp32.safetensors',
        'minimax_h3_fl2va_pruned_int8_convrot.safetensors',
        REF2VA_FILE,'minimax_h3_video_vae_fp16.safetensors',
        'minimax_h3_turbo_v4_step600_ema.safetensors']

# 只列出本次啟用的功能；此時還不會下載 H3、Qwen 等大模型。
selected=set(COMMON)
qwen_shared=['qwen3vl_8b_int8_convrot.safetensors','qwen_image_2.1_vae_bf16.safetensors']
QWEN_AVAILABLE={}
if ENABLE_QWEN:
    QWEN_AVAILABLE['original']='qwen_image_2.1_int8_convrot.safetensors'
    selected.update(qwen_shared + [QWEN_AVAILABLE['original']])
    if ENABLE_NOCTQ:
        QWEN_AVAILABLE['noct-q-v4']='NoctQ_V4_int8_convrot.safetensors'
        selected.add(QWEN_AVAILABLE['noct-q-v4'])
# REDQW21 requires Civitai authorization. If user already supplied it in Drive,
# expose it via a symlink (no copy); otherwise never auto-download it.
REDQW21_SOURCE=DRIVE_MODELS_ROOT/'diffusion_models/REDQW21_V2_int8.safetensors'
if ENABLE_QWEN and REDQW21_SOURCE.is_file():
    REDQW21_LINK=MODELS_ROOT/'diffusion_models/REDQW21_V2_int8.safetensors'
    REDQW21_LINK.parent.mkdir(parents=True,exist_ok=True)
    if REDQW21_LINK.is_symlink(): REDQW21_LINK.unlink()
    if not REDQW21_LINK.exists(): REDQW21_LINK.symlink_to(REDQW21_SOURCE)
    QWEN_AVAILABLE['redqw21-v2']='REDQW21_V2_int8.safetensors'
SWAP_AVAILABLE={}

if ENABLE_SWAP:
    SWAP_AVAILABLE={'head':'minimax_h3_head_swap_v1.0_r32.safetensors',
                    'character':'minimax_h3_character_swap_v1.0_ref2va_r64_gl.safetensors',
                    'akatz-v1.5':'h3_character_swap_v1.5.safetensors'}
    if globals().get('ENABLE_UNTMODS_FACESWAP',True):
        SWAP_AVAILABLE['untmods-face']='SS_FaceSwap_MiniMax_H3_REF2VA.safetensors'
        selected.add('minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors')
    selected.update(SWAP_AVAILABLE.values())
HAS_SEED=bool(ENABLE_SEEDVR2)
if HAS_SEED:
    selected.update(['seedvr2_3b_int8_convrot.safetensors','seedvr2_ema_vae_fp16.safetensors'])
HAS_CAMERA=bool(ENABLE_CAMERA and QWEN_AVAILABLE)
if HAS_CAMERA: selected.add('qwen_image_2.1_multiple_angles_v2_step1500.safetensors')
if ENABLE_ORB360: selected.add('minimax_h3_orb360_step1500.safetensors')
# mk27：第 9 格檢查需要這兩個名稱（mk26 漏定義）。
SINGULARITY_MODEL='Minimax-h3_Singularity_ref2va_v2_pruned_int8.safetensors'
SINGULARITY_LORA='minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors'
HAS_SINGULARITY=bool(INSTALL_SINGULARITY)
if HAS_SINGULARITY:
    selected.update([SINGULARITY_MODEL,SINGULARITY_LORA])
# mk27：H3-Studio 後端（修臉就緒檢查）寫死 W4A8 Ref2VA 檔名；mk25 因 Drive 兩檔都在而通過。
# 兩種 Ref2VA 量化都建佔位檔；未選用的那個只有真的被工作流載入時才會下載。
selected.update(REF2VA_FILES.values())
assert all(n in CATALOG for n in selected), '模型清單缺項：'+str(selected-set(CATALOG))

# File placeholders are used for UI dropdown enumeration only (0 bytes, never loaded).
ACTIVE_MODELS=MODELS_ROOT
manifest={}
for name in sorted(selected):
    item=CATALOG[name]
    file=ACTIVE_MODELS/item['folder']/name
    file.parent.mkdir(parents=True,exist_ok=True)
    if not file.exists(): file.touch()
    manifest[(Path(item['folder'])/name).as_posix()]={'url':item['url'],'sha256':item['sha256']}

# Preserve support for user-provided custom LoRAs directly from Drive. No Drive copy involved.
extra_loras={}
if ENABLE_DRIVE_CUSTOM_LORAS:
    personal=DRIVE_MODELS_ROOT/'loras'
    if personal.is_dir():
        for file in personal.rglob('*.safetensors'):
            if file.name in CATALOG or not file.is_file(): continue
            extra_loras[file.relative_to(personal).as_posix()]=file
cfg={'h3_colab':{'base_path':str(ACTIVE_MODELS)}}
for folder in ['diffusion_models','text_encoders','vae','loras']:
    cfg['h3_colab'][folder]=folder
if extra_loras:
    cfg['h3_user_loras']={'base_path':str(DRIVE_MODELS_ROOT),'loras':'loras'}
(COMFY/'extra_model_paths.yaml').write_text(json.dumps(cfg,indent=2),encoding='utf-8')

# Write the lazy on-demand direct-download hook into ComfyUI before its first start.
legacy_cache_node=COMFY/'custom_nodes/h3_colab_ssd_cache'
if legacy_cache_node.exists():
    shutil.rmtree(legacy_cache_node)   # Prevent mk25 Drive-copy hook from coexisting with mk26.
node_dir=COMFY/'custom_nodes/h3_colab_direct_download'
node_dir.mkdir(parents=True,exist_ok=True)
(node_dir/'catalog.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
DOWNLOAD_HELPER='import hashlib\nimport os\nimport shutil\nimport time\nfrom pathlib import Path\nimport requests\n\n\ndef download_verified(url, target, expected_sha256=None, report=print, reserve_gib=8, attempts=4):\n    """Download directly to SSD; resumable Range + full SHA256; never expose partial file as a model."""\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    part = target.with_name(target.name + \'.part\')\n    started = time.monotonic()\n    for attempt in range(1, attempts + 1):\n        try:\n            have = part.stat().st_size if part.is_file() else 0\n            headers = {\'Range\': f\'bytes={have}-\'} if have else {}\n            with requests.get(url, stream=True, headers=headers, timeout=(30, 300), allow_redirects=True) as response:\n                if have and response.status_code == 416:\n                    # It may already be complete: try hash validation below.\n                    response.close()\n                    completed = True\n                    total = have\n                else:\n                    response.raise_for_status()\n                    if have and response.status_code == 200:\n                        report(\'[H3 HF download] server did not honor Range; restarting \' + target.name)\n                        have = 0\n                        part.unlink(missing_ok=True)\n                    if have and response.status_code != 206:\n                        raise OSError(f\'Range resume expected HTTP 206, got {response.status_code}\')\n                    if have:\n                        content_range = response.headers.get(\'Content-Range\', \'\')\n                        if not content_range.startswith(f\'bytes {have}-\'):\n                            raise OSError(f\'Unexpected Content-Range: {content_range}\')\n                    remaining = int(response.headers.get(\'Content-Length\') or 0)\n                    total = have + remaining if remaining else 0\n                    free = shutil.disk_usage(target.parent).free\n                    if total and free < max(0, total - have) + int(reserve_gib * 1024**3):\n                        raise OSError(f\'Colab SSD 空間不足：還需約 {(total-have)/1024**3:.1f} GiB ＋ {reserve_gib} GiB 保留空間\')\n                    downloaded = have\n                    last_print_at = time.monotonic()\n                    last_print_bytes = downloaded\n                    next_mark = downloaded + 256 * 1024**2\n                    report(f\'[H3 HF download] {target.name}: HTTP {response.status_code}, \'\n                           f\'{have/1024**3:.2f} GiB already downloaded; source={url.split("/resolve/")[0]}\')\n                    with part.open(\'ab\' if have else \'wb\') as f:\n                        for chunk in response.iter_content(chunk_size=4 * 1024**2):\n                            if not chunk:\n                                continue\n                            f.write(chunk)\n                            downloaded += len(chunk)\n                            now = time.monotonic()\n                            if downloaded >= next_mark or now - last_print_at >= 5:\n                                instant = (downloaded - last_print_bytes) / max(.001, now - last_print_at) / 1024**2\n                                avg = (downloaded - have) / max(.001, now - started) / 1024**2\n                                percent = f\' ({100*downloaded/total:.1f}%)\' if total else \'\'\n                                report(f\'[H3 HF download] {target.name}: {downloaded/1024**3:.2f}\'\n                                       f\'/{total/1024**3:.2f} GiB{percent}, \'\n                                       f\'即時 {instant:.1f} MiB/s, 平均 {avg:.1f} MiB/s\')\n                                last_print_at, last_print_bytes = now, downloaded\n                                next_mark = downloaded + 256 * 1024**2\n                    completed = True\n                    if total and downloaded != total:\n                        raise OSError(f\'Download size mismatch: {downloaded} != {total}\')\n            if not completed or not part.is_file():\n                raise OSError(\'Missing downloaded file\')\n            report(\'[H3 HF download] SHA256 verifying: \' + target.name)\n            h = hashlib.sha256()\n            with part.open(\'rb\') as src:\n                for chunk in iter(lambda: src.read(16 * 1024**2), b\'\'):\n                    h.update(chunk)\n            digest = h.hexdigest()\n            if expected_sha256 and digest.lower() != expected_sha256.lower():\n                part.unlink(missing_ok=True)\n                raise ValueError(f\'SHA256 mismatch: {target.name}; damaged file removed\')\n            if part.stat().st_size == 0:\n                part.unlink(missing_ok=True)\n                raise ValueError(f\'Empty downloaded file: {target.name}\')\n            os.replace(part, target)\n            secs = time.monotonic() - started\n            report(f\'[H3 HF download] READY: {target.name} ({target.stat().st_size/1024**3:.2f} GiB), \'\n                   f\'{secs:.1f}s total, {target.stat().st_size/max(.001,secs)/1024**2:.1f} MiB/s overall\')\n            return digest\n        except ValueError:\n            raise\n        except Exception as err:\n            report(f\'[H3 HF download] 第 {attempt}/{attempts} 次失敗: {target.name}: {err}\')\n            if attempt == attempts:\n                raise\n            time.sleep(min(30, 5 * attempt))\n    raise RuntimeError(\'All attempts exhausted\')\n'
(node_dir/'_download.py').write_text(DOWNLOAD_HELPER,encoding='utf-8')
NODE_SOURCE="# mk26: lazy downloads from original Hugging Face/author URLs into Colab SSD.\n# Zero-byte placeholders allow model names to be displayed in ComfyUI dropdowns;\n# they are replaced atomically with verified weights ONLY when a workflow executes.\nimport json\nimport logging\nimport os\nimport sys\nimport threading\nfrom pathlib import Path\nimport folder_paths\nfrom ._download import download_verified\n\nBASE = Path(__BASE__)\nMANIFEST = json.loads((Path(__file__).parent / 'catalog.json').read_text(encoding='utf-8'))\nEXECUTION = os.path.normcase(os.path.join(os.path.dirname(os.path.abspath(folder_paths.__file__)), 'execution.py'))\n_lock = threading.Lock()\n_original = folder_paths.get_full_path\n\n\ndef _executing():\n    frame = sys._getframe(1)\n    while frame:\n        if os.path.normcase(os.path.abspath(frame.f_code.co_filename)) == EXECUTION:\n            return True\n        frame = frame.f_back\n    return False\n\n\ndef _get_model_path(path):\n    if not path:\n        return path\n    file = Path(path)\n    try:\n        relative = file.relative_to(BASE).as_posix()\n    except ValueError:\n        return path  # User-supplied LoRAs and non-catalog assets are not changed.\n    item = MANIFEST.get(relative)\n    if not item:\n        return path\n    if file.is_file() and file.stat().st_size > 0:\n        return path\n    if not _executing():\n        return path  # Node dropdowns and discovery never trigger downloads.\n    with _lock:\n        if file.is_file() and file.stat().st_size > 0:\n            return str(file)\n        # Failed downloads never let the original zero-byte placeholder reach the model loader.\n        digest = download_verified(item['url'], file, item['sha256'], report=logging.info,\n                                   reserve_gib=__RESERVE__)\n        if not item['sha256']:\n            logging.info(f'[H3 HF download] {file.name} 未釘選 SHA256（未驗證）；實際 SHA256={digest}')\n    return str(file)\n\n\ndef get_full_path(folder_name, filename):\n    return _get_model_path(_original(folder_name, filename))\n\nfolder_paths.get_full_path = get_full_path\n\n\n# mk29: FlashVSR / VOSR weights are downloaded by a background process (h3_bg_models).\n# A prompt that uses them waits here (in the prompt worker thread, not the web server) until that\n# download finishes; if the downloader is gone, the prompt runs and the node reports what is missing.\nimport time\nimport execution\n\nBG = Path(__BG__)\nBG_PID = BG.parent / 'h3_bg_models.pid.json'\nGATES = {'FlashVSRNode': 'flashvsr', 'FlashVSRNodeAdv': 'flashvsr', 'FlashVSRInitPipe': 'flashvsr', 'H3StudioVOSR': 'vosr'}\n\n\ndef _bg_downloading(name):\n    try:\n        if not (BG / (name + '.pending')).exists():\n            return False\n        saved = json.loads(BG_PID.read_text())\n        stat = Path(f\"/proc/{saved['pid']}/stat\").read_text().rsplit(')', 1)[1].split()\n        return stat[0] != 'Z' and stat[19] == saved['token']\n    except Exception:\n        return False\n\n\ndef _wait_background_models(prompt):\n    names = sorted({GATES[n.get('class_type')] for n in prompt.values()\n                    if isinstance(n, dict) and n.get('class_type') in GATES})\n    for name in names:\n        deadline = time.monotonic() + 2 * 3600\n        told = False\n        while _bg_downloading(name) and time.monotonic() < deadline:\n            if not told:\n                logging.info(f'[H3 bg models] 等待 {name} 背景下載完成後再執行（進度見 /content/h3_bg_models.log）')\n                told = True\n            time.sleep(2)\n        if told:\n            logging.info(f'[H3 bg models] {name} 等待結束，開始執行')\n\n\n_original_execute = execution.PromptExecutor.execute\n\n\ndef _gated_execute(self, prompt, *args, **kwargs):\n    try:\n        _wait_background_models(prompt)\n    except Exception as err:\n        logging.warning(f'[H3 bg models] wait skipped: {err}')\n    return _original_execute(self, prompt, *args, **kwargs)\n\n\nexecution.PromptExecutor.execute = _gated_execute\n\nNODE_CLASS_MAPPINGS = {}\n"
(node_dir/'__init__.py').write_text(NODE_SOURCE.replace('__BASE__',repr(str(ACTIVE_MODELS)))
                                        .replace('__RESERVE__',str(MODEL_DOWNLOAD_RESERVE_GIB))
                                        .replace('__BG__',repr(str(ROOT/'h3_bg_models'))),encoding='utf-8')
# Reuse exactly the same verified downloader for the independent FlashVSR / VOSR installers.
exec(compile(DOWNLOAD_HELPER, str(node_dir/'_download.py'),'exec'),globals())

print('mk26：首次載入時下載到 SSD（不是從 Drive copy），下載進度見 /content/h3_comfy.log。',flush=True)
print('SSD 目錄:',ACTIVE_MODELS,'模型選項:',len(manifest),flush=True)
print('Qwen:',list(QWEN_AVAILABLE),'Swap:',list(SWAP_AVAILABLE),'SeedVR2:',HAS_SEED,
      'Camera:',HAS_CAMERA,'Singularity:',HAS_SINGULARITY,'自訂 Drive LoRA:',len(extra_loras),flush=True)


In [ ]:
# 6. 修臉：與本機相同節點版本，Linux 安裝；僅規則／手動追蹤，不需 InsightFace。
if INSTALL_FACE_REFINE:
    checkout('https://github.com/Carasibana/ComfyUI-H3-FaceRefine.git',COMFY/'custom_nodes/ComfyUI-H3-FaceRefine',FACE_REFINE_COMMIT)
    audio=COMFY/'custom_nodes/ComfyUI-H3-NativeAudioLock'
    audio.mkdir(parents=True,exist_ok=True)
    url='https://raw.githubusercontent.com/Shrek3OnVH5/MiniMax-H3-NativeAudio-MusicVideo-Workflow/84895cfaaef39abb42fb844b02cb42f868c7bebf/custom_nodes/ComfyUI-H3-NativeAudioLock/__init__.py'
    run(['curl','--fail','--location','--retry','3',url,'-o',audio/'__init__.py'])
    if not ENV_FROM_SNAPSHOT:  # mk28：快照已含這些套件
        run([PYTHON,'-m','pip','uninstall','-y','opencv-python-headless'])
        run([PYTHON,'-m','pip','install','opencv-python==5.0.0.93'])
        run([PYTHON,'-m','pip','install','--no-deps','ultralytics==8.4.178','ultralytics-thop==2.2.2','ultralytics-platform==0.1.90','scenedetect==0.7.2','polars==2.0.0','polars-runtime-32==2.0.0','platformdirs==4.13.3','nvidia-ml-py==13.615.71'])
    detector=MODELS_ROOT/'ultralytics/bbox/face_yolov8m.pt'
    detector.parent.mkdir(parents=True,exist_ok=True)
    expected='717923c19b3f4bbf5250b728f1fa6b2cb72a33aed1d236ea9caf0e21ad943e5f'
    if not detector.is_file() or hashlib.sha256(detector.read_bytes()).hexdigest()!=expected:
        print('從原始 Hugging Face 網址下載 face_yolov8m.pt → Colab SSD',flush=True)
        download_verified('https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt',
                          detector,expected,report=lambda msg: print(msg,flush=True),reserve_gib=MODEL_DOWNLOAD_RESERVE_GIB)
    assert hashlib.sha256(detector.read_bytes()).hexdigest()==expected, '偵測器雜湊不符'
    local=COMFY/'models/ultralytics/bbox/face_yolov8m.pt'; local.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(detector,local)
    run([PYTHON,'-c','from ultralytics import YOLO; import scenedetect; print("FaceRefine dependencies ready")'])
# mk29：FlashVSR／VOSR 只裝節點；模型交給背景程式（第 8 格啟動），用到時若還沒下載完會等待。
stop_managed('h3_bg_models')
BG_DIR=ROOT/'h3_bg_models'; BG_DIR.mkdir(exist_ok=True)
for _m in BG_DIR.glob('*'): _m.unlink()
BG_JOBS={}
# FlashVSR: optional engine. Failure only disables the option.
HAS_FLASHVSR=False
if INSTALL_FLASHVSR:
    try:
        checkout('https://github.com/lihaoyun6/ComfyUI-FlashVSR_Ultra_Fast.git',COMFY/'custom_nodes/ComfyUI-FlashVSR_Ultra_Fast',FLASHVSR_COMMIT)
        if not ENV_FROM_SNAPSHOT: run([PYTHON,'-m','pip','install','-q','einops','huggingface_hub','safetensors','tqdm'])
        need=['LQ_proj_in.ckpt','TCDecoder.ckpt','diffusion_pytorch_model_streaming_dmd.safetensors','Wan2.1_VAE.pth']
        ssd_fv=MODELS_ROOT/'FlashVSR-v1.1'; ssd_fv.mkdir(parents=True,exist_ok=True)
        fv_dir=COMFY/'models/FlashVSR-v1.1'
        if fv_dir.is_symlink(): fv_dir.unlink()
        elif fv_dir.exists(): shutil.rmtree(fv_dir)
        fv_dir.symlink_to(ssd_fv,target_is_directory=True)
        if all((ssd_fv/n).is_file() for n in need):
            print('FlashVSR 就緒（模型已在 Colab SSD）',flush=True)
        else:
            BG_JOBS['flashvsr']={'dir':str(ssd_fv),'repo':'JunhaoZhuang/FlashVSR-v1.1','need':need}
            (BG_DIR/'flashvsr.pending').touch()
            print('FlashVSR 節點已安裝；模型（約 6 GiB）將在背景下載',flush=True)
        HAS_FLASHVSR=True
    except Exception as exc:
        print('⚠ FlashVSR 安裝失敗，畫質提升只提供 SeedVR2：',type(exc).__name__,str(exc)[:300],flush=True)
# VOSR 1x refinement. Original upstream assets downloaded in the background to Colab SSD.
HAS_VOSR=False
if INSTALL_VOSR:
    try:
        checkout('https://github.com/ylchen333/ComfyUI-VOSR2.git',COMFY/'custom_nodes/ComfyUI-VOSR2',VOSR_COMMIT)
        if not ENV_FROM_SNAPSHOT: run([PYTHON,'-m','pip','install','-q','-r',COMFY/'custom_nodes/ComfyUI-VOSR2/requirements.txt'])
        hf='https://huggingface.co/CSWRY/VOSR/resolve/f24b3061b7f350b81e1907bdcfc27f71fb4ff3f3/'
        bundle=MODELS_ROOT/'vosr2/VOSR2'; bundle.mkdir(parents=True,exist_ok=True)
        vosr_files=[('VOSR2/args.json','args.json',None,0.0),
                    ('VOSR2/checkpoints/ema_model.safetensors','checkpoints/ema_model.safetensors','bdcaa81e4c675b6074de643e27daafe73eb8125d4d0264d1bf30a004e9644b71',5.2),
                    ('Qwen-Image-vae-2d/config.json','Qwen-Image-vae-2d/config.json',None,0.0),
                    ('Qwen-Image-vae-2d/diffusion_pytorch_model.safetensors','Qwen-Image-vae-2d/diffusion_pytorch_model.safetensors','1dd2c67f3f29734d48709513ed181d4f351966d76ba95f77a5cff8e1daf547fa',0.2)]
        dino=bundle/'dinov2_vitl14.safetensors'
        need=sum(g for _,rel,_,g in vosr_files if not (bundle/rel).is_file())+(1.2 if not dino.is_file() else 0)
        if need:
            free=shutil.disk_usage(MODELS_ROOT).free/1024**3
            assert free>need+MODEL_DOWNLOAD_RESERVE_GIB, f'VOSR SSD 空間不足：需要約 {need:.1f} GiB + 保留 {MODEL_DOWNLOAD_RESERVE_GIB} GiB'
        link=COMFY/'models/vosr2'
        if link.is_symlink(): link.unlink()
        elif link.exists():
            # ComfyUI-VOSR2 creates an empty models/vosr2 on import; never delete real files.
            assert not any(p.is_file() for p in link.rglob('*')), f'{link} 內已有檔案，請先移出模型目錄'
            shutil.rmtree(link)
        link.symlink_to(MODELS_ROOT/'vosr2',target_is_directory=True)
        if need:
            BG_JOBS['vosr']={'bundle':str(bundle),'hf':hf,'files':vosr_files,'dino':str(dino),
                             'dino_tmp':str(ROOT/'h3_vosr_tmp/dinov2_vitl14_pretrain.pth'),
                             'dino_sha':'d5383ea8f4877b2472eb973e0fd72d557c7da5d3611bd527ceeb1d7162cbf428'}
            (BG_DIR/'vosr.pending').touch()
            print(f'VOSR 節點已安裝；模型（約 {need:.1f} GiB）將在背景下載',flush=True)
        else:
            print('VOSR 就緒（模型已在 Colab SSD）',flush=True)
        HAS_VOSR=True
    except Exception as exc:
        print('⚠ VOSR 安裝失敗，畫質提升不提供 VOSR：',type(exc).__name__,str(exc)[:300],flush=True)
BG_SCRIPT=r"""
import json, os, shutil, subprocess, sys, time
from pathlib import Path
cfg=json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
sys.path.insert(0,cfg['helper_dir'])
from _download import download_verified
BG=Path(cfg['bg_dir']); reserve=cfg['reserve']
def log(msg): print(msg,flush=True)

def flashvsr(job):
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=job['repo'],local_dir=job['dir'])
    missing=[n for n in job['need'] if not (Path(job['dir'])/n).is_file()]
    if missing: raise RuntimeError('FlashVSR 模型缺檔：'+', '.join(missing))

def vosr(job):
    bundle=Path(job['bundle'])
    for remote,rel,sha,_ in job['files']:
        if not (bundle/rel).is_file():
            download_verified(job['hf']+remote,bundle/rel,sha,report=log,reserve_gib=reserve)
    json.loads((bundle/'args.json').read_text(encoding='utf-8'))
    dino=Path(job['dino'])
    if not dino.is_file():
        # Upstream ships a torch pickle; convert locally, keep safetensors on Colab SSD.
        pth=Path(job['dino_tmp'])
        download_verified(job['hf']+'torch_cache/checkpoints/dinov2_vitl14_pretrain.pth',pth,job['dino_sha'],report=log,reserve_gib=reserve)
        local=pth.with_name('dinov2_vitl14.safetensors')
        subprocess.run([sys.executable,'-c','import sys,torch\nfrom safetensors.torch import save_file\nstate=torch.load(sys.argv[1],map_location="cpu",weights_only=True,mmap=True)\nsave_file({k:v.contiguous() for k,v in state.items()},sys.argv[2])',str(pth),str(local)],check=True)
        part=dino.with_name(dino.name+'.part'); os.replace(local,part); os.replace(part,dino)
        shutil.rmtree(pth.parent)
        log('DINOv2 已轉換並存入 Colab SSD: '+str(dino))

for name,fn in (('vosr',vosr),('flashvsr',flashvsr)):
    job=cfg['jobs'].get(name)
    if not job or not (BG/(name+'.pending')).exists(): continue
    t=time.monotonic(); log(f'[bg] {name} 開始下載')
    try:
        fn(job)
        (BG/(name+'.ready')).write_text('ok\n')
        log(f'[bg] {name} READY（{time.monotonic()-t:.0f} 秒）')
    except Exception as exc:
        (BG/(name+'.failed')).write_text(f'{type(exc).__name__}: {exc}\n')
        log(f'[bg] {name} FAILED: {type(exc).__name__}: {exc}')
    finally:
        (BG/(name+'.pending')).unlink(missing_ok=True)
log('[bg] DONE')
"""
(ROOT/'h3_bg_models.py').write_text(BG_SCRIPT,encoding='utf-8')
(ROOT/'h3_bg_models.json').write_text(json.dumps({'helper_dir':str(COMFY/'custom_nodes/h3_colab_direct_download'),'bg_dir':str(BG_DIR),
    'reserve':MODEL_DOWNLOAD_RESERVE_GIB,'jobs':BG_JOBS},ensure_ascii=False,indent=1),encoding='utf-8')
if BG_JOBS: print('背景下載排程：',', '.join(BG_JOBS),'（第 8 格啟動，日誌 /content/h3_bg_models.log）',flush=True)
# ultralytics-platform 0.1.90 requires httpx (confirmed missing on Colab).
if not ENV_FROM_SNAPSHOT: run([PYTHON,'-m','pip','install','httpx'])
run([PYTHON,'-m','pip','check'])


In [ ]:
# 7. 只套用到 Colab checkout；固定版本與精確匹配避免改錯位置。
import re
def pristine(rel):
    return subprocess.check_output(['git','-C',REPO,'show',STUDIO_COMMIT+':'+rel],text=True,encoding='utf-8')
def replace(text,old,new,count=1):
    assert text.count(old)==count, f'Colab patch mismatch: {old[:100]} ({text.count(old)} != {count})'
    return text.replace(old,new)
def drop_sage(text,name,expected):
    # mk24: replace every SageAttention patch node with the Comfy Kitchen attention backend (no sageattention package needed).
    text,n=re.subn(r'node\("MiniMaxH3MemoryEfficientSageAttentionPatch", \{ model: (.+?) \}(?:, "[^"]*")?\)',
        lambda m:'node("ModelAttentionBackend", { model: '+m.group(1)+', attention: "comfy kitchen attention" }, "Comfy Kitchen 注意力後端")',text)
    assert n==expected and 'SageAttentionPatch' not in text, f'{name}: Sage 節點替換數量 {n} != {expected}'
    return text
def save(rel,text):
    (REPO/rel).write_text(text,encoding='utf-8')
for rel in ('lib/comfy.ts','lib/tts.ts','lib/drama3.ts'):
    path='h3-local-studio/'+rel
    s=replace(pristine(path),'http://127.0.0.1:8188','/comfy')
    if rel=='lib/comfy.ts':
        # Ref2VA main model follows REF2VA_QUANT (cell 1/5). Fail loudly if the Studio source changed.
        _w4='minimax_h3_ref2va_pruned_w4a8_mixed.safetensors'
        assert s.count(_w4)>=1, 'lib/comfy.ts 找不到 Ref2VA W4A8 檔名，Studio 原始碼已變更，請檢查 Ref2VA 載入段落'
        if REF2VA_FILE!=_w4: s=s.replace(_w4,REF2VA_FILE)
        # Colab: datacenter GPU needs no per-step cooldown. Default 0, minimum 0.
        s=replace(s,'export const MIN_COOLDOWN_SECONDS = 5;','export const MIN_COOLDOWN_SECONDS = 0;')
        s=replace(s,'export const DEFAULT_COOLDOWN_SECONDS = 15;','export const DEFAULT_COOLDOWN_SECONDS = 0;')
        # mk24: sol-attn only. No SageAttention patch node anywhere; Ref2VA uses LongMedia's embedded sol mode.
        s=replace(s,'  if (options.attentionMethod === "sol") throw new Error("參考生影片的 LongMedia 記憶體分塊與 sol-attn 尚不相容，請使用 SageAttention。");\n','')
        s=replace(s,'attention_mode: "existing"','attention_mode: "sol"')
        s=replace(s,'  if (options.attentionMethod === "sol") {\n','  {\n')
        s=replace(s,'options.attentionMethod ?? source?.attentionMethod ?? "sage"','"sol" as const')
        s=replace(s,'? "sol" as const : "sage" as const','? "sol" as const : "sol" as const')
        s=drop_sage(s,'lib/comfy.ts',3)
        # mk25: 12/16/20-step native QUALITY, all three video modes (not Turbo LoRA).
        # Separate profile identifiers preserve each choice in works metadata, redo and mode settings.
        s=replace(s,'  profile: "fast" | "cooled-fast" | "cooled-turbo-4" | "cooled-turbo-8" | "quality" | "safe-long" | "low-vram";',
                    '  profile: "fast" | "cooled-fast" | "cooled-turbo-4" | "cooled-turbo-8" | "quality-12" | "quality-16" | "quality" | "safe-long" | "low-vram";')
        # Text-to-video and first/last-frame workflows share this native QUALITY path.
        s=replace(s,'  const quality = options.profile === "quality";',
                    '  const quality = options.profile === "quality" || options.profile === "quality-12" || options.profile === "quality-16";\n  const nativeSteps = options.profile === "quality-12" ? 12 : options.profile === "quality-16" ? 16 : 20;')
        s=replace(s,'steps: quality ? 20 : cooledTurbo8 ? 8 : cooledTurbo4 ? 4 : 6',
                    'steps: quality ? nativeSteps : cooledTurbo8 ? 8 : cooledTurbo4 ? 4 : 6')
        # The UI cooldown indicator must report actual steps for reference mode too.
        s=replace(s,'  return profile === "cooled-turbo-4" ? 4 : profile === "fast" || profile === "cooled-fast" ? 6 : 8;',
                    '  return profile === "quality-12" ? 12 : profile === "quality-16" ? 16 : profile === "quality" ? 20 : profile === "cooled-turbo-4" ? 4 : profile === "fast" || profile === "cooled-fast" ? 6 : 8;')
        # Ref2VA is a DIFFERENT workflow. Native mode must bypass both Turbo LoRA
        # and Turbo sampler while retaining LongMedia's references and segmenting.
        ref_start=s.index('export function buildReferenceWorkflow(')
        ref_end=s.index('/**\n * Single-frame image pipeline.',ref_start)
        ref=s[ref_start:ref_end]
        ref=replace(ref,'  const continuation = Boolean(continuationFrame);\n',
                    '  const continuation = Boolean(continuationFrame);\n  const quality = options.profile === "quality" || options.profile === "quality-12" || options.profile === "quality-16";\n')
        ref=replace(ref,'steps: referenceVideoSteps(options.profile), denoise: 1 }, `Turbo ${referenceVideoSteps(options.profile)} 步排程`',
                    'steps: referenceVideoSteps(options.profile), denoise: 1 }, `${quality ? "原生" : "Turbo"} ${referenceVideoSteps(options.profile)} 步排程`')
        ref=replace(ref,'"11": node("VAELoader", { vae_name: "minimax_h3_video_vae_int8_convrot.safetensors" }, "影片 VAE")',
                    '"11": node("VAELoader", { vae_name: quality ? "minimax_h3_video_vae_fp16.safetensors" : "minimax_h3_video_vae_int8_convrot.safetensors" }, "影片 VAE")')
        ref=replace(ref,'"119": node("ModelAttentionBackend", { model: ["120", 0], attention: "comfy kitchen attention" }, "Comfy Kitchen 注意力後端")',
                    '"119": node("ModelAttentionBackend", { model: quality ? ["6", 0] : ["120", 0], attention: "comfy kitchen attention" }, "Comfy Kitchen 注意力後端")')
        ref=replace(ref,'  const extras = options.extraLoras ?? [];\n', '''  if (quality) {
    // Native sampling: base Ref2VA UNET, RES Multistep, simple sigma scheduler.
    // Keep the LongMedia sampler and its built-in sol-attn / memory controls.
    delete graph["120"];
    graph["121"] = node("KSamplerSelect", { sampler_name: "res_multistep" }, "H3 原生採樣器");
    graph["122"] = node("H3CooledSampler", { sampler: ["121", 0], seconds: cooldown }, "H3 原生冷卻採樣器");
  }

  const extras = options.extraLoras ?? [];
''')
        ref=replace(ref,'    graph["120"].inputs.low_vram = true;\n    graph["119"].inputs.model = stackExtraLoras(graph, extras, ["120", 0]);',
                    '    if (!quality) graph["120"].inputs.low_vram = true;\n    graph["119"].inputs.model = stackExtraLoras(graph, extras, quality ? ["6", 0] : ["120", 0]);')
        s=s[:ref_start]+ref+s[ref_end:]
        # Continuations of Ref2VA must retain the selected native profile.
        s=replace(s,'source.profile === "low-vram" ? "low-vram" : source.profile === "fast"',
                    '(source.profile === "quality" || source.profile === "quality-12" || source.profile === "quality-16") ? source.profile : source.profile === "low-vram" ? "low-vram" : source.profile === "fast"')
        # A finished ComfyUI result must win over a late/stray stop flag ("工作已停止" after a completed face job).
        s=replace(s,'    if (stopped?.()) throw new Error("工作已停止。");\n','')
        s=replace(s,'    const queueResponse = await request("/queue");','    if (stopped?.()) throw new Error("工作已停止。");\n    const queueResponse = await request("/queue");')
        # Colab tunnel: transient network drops abort whole jobs ("Failed to fetch"). Retry idempotent GETs.
        s=replace(s,'  const response = await fetch(`${COMFY_URL}${path}`, init);','''  const method = (init?.method ?? "GET").toUpperCase();
  let response: Response;
  for (let attempt = 0; ; attempt++) {
    try {
      response = await fetch(`${COMFY_URL}${path}`, init);
      if (method === "GET" && [502, 503, 504, 520, 522, 523, 524, 530].includes(response.status) && attempt < 40) throw new Error(`tunnel ${response.status}`);
      break;
    } catch (error) {
      if (method !== "GET" || attempt >= 40) throw error;
      await new Promise((resolve) => setTimeout(resolve, 3000));
    }
  }''')
    save(path,s)
save('h3-local-studio/lib/vosr.ts',replace(pristine('h3-local-studio/lib/vosr.ts'),'http://127.0.0.1:8188/view','/comfy/view'))
page=pristine('h3-local-studio/app/page.tsx')
if not QWEN_AVAILABLE: page=replace(page,'    { id: "edit", icon: "✎", label: "圖像編輯" },\n','')
if not GEMINI_KEY: page=replace(page,'    { id: "tts", icon: "♫", label: "語音生成" },\n','')
page,_n=re.subn(r'[ \t]*<label className="attention-control">.*?</label>\n','',page,flags=re.S); assert _n==1,'找不到注意力選單'
page=replace(page,'attentionMethod: sourceMode === "reference" ? "sage" : attentionMethod,','attentionMethod: "sol",')
page=replace(page,'video.attentionMethod ?? "sage"','"sol"',2)
page=replace(page,'{sourceMode === "reference" ? <span> · 參考生影片使用 SageAttention，sol-attn 尚不相容 LongMedia 記憶體分塊</span> : attentionMethod === "sol" && <span>','{sourceMode === "reference" ? <span> · 參考生影片使用 LongMedia 內建 sol 模式</span> : <span>')
page=replace(page,'disabled={!video.extendable}','disabled={true} title="Colab 停用延伸"')
page=replace(page,'disabled={deleting === key}','disabled={true} title="Colab 停用刪除"',2)
page=replace(page,'disabled={deleting === `${audio.subfolder}/${audio.filename}`}','disabled={true} title="Colab 停用刪除"')
page=replace(page,'{video.extendable && <small>','{false && <small>')
# Remove mode tabs that cannot run with this runtime/model set.
lines=page.splitlines(keepends=True)
labels=['RTX提升']+([] if SWAP_AVAILABLE else ['影片換角'])+([] if HAS_SEED or HAS_VOSR else ['影片畫質提升'])
for label in labels:
    matches=[line for line in lines if line.strip().startswith('<button ') and '>'+label+'</button>' in line]
    assert len(matches)==1, label
    lines.remove(matches[0])
page=''.join(lines)
page=replace(page,'disabled={!connected || isOptimizing || Boolean(continuationSource)}','disabled={true} title="Colab 未配置 Codex／Grok 優化服務"')
page=replace(page,'disabled={!connected || isOptimizing}','disabled={true} title="Colab 未配置 Codex 優化服務"')
if not INSTALL_FACE_REFINE:
    page=replace(page,'checked={faceEnabled} onChange=', 'checked={faceEnabled} disabled title="未安裝修臉功能" onChange=')
    page=replace(page,'<button role="menuitem" onClick={() => { setOpenMenu(null); onRefine(video); }}>','<button disabled title="未安裝修臉功能" role="menuitem" onClick={() => { setOpenMenu(null); onRefine(video); }}>')
if not HAS_CAMERA:
    page=replace(page,'checked={qwenCameraEnabled} onChange=', 'checked={qwenCameraEnabled} title="缺少攝影機 LoRA" disabled={true} onChange=')
    # Avoid duplicate disabled attributes on the same input.
    page=page.replace('setQwenMention(null); }} disabled={isOptimizing} />','setQwenMention(null); }} />')
if QWEN_AVAILABLE:
    default=next(iter(QWEN_AVAILABLE))
    page=replace(page,'useState<QwenModel>("original")','useState<QwenModel>('+json.dumps(default)+')')
    page=replace(page,'Object.entries(QWEN_MODELS).map','Object.entries(QWEN_MODELS).filter(([key]) => '+json.dumps(list(QWEN_AVAILABLE))+'.includes(key)).map')
page=replace(page,'文字、首尾幀或多圖參考，在你的 RTX 4080 Laptop 上生成有聲影片。','文字、首尾幀或多圖參考，在 Colab GPU 生成影片，完成後自動備份到 Google Drive。')
# Colab: offer 768P (native H3, aspect-scaled) for text and first/last-frame modes too.
page=replace(page,'                          <option value="p540">{sizeLabels.p540} · 540P</option>\n                        </>\n                      )}',
             '                          <option value="p540">{sizeLabels.p540} · 540P</option>\n                          <option value="native">{sizeLabels.native} · 768P</option>\n                        </>\n                      )}')
# Face-refine stop buttons: confirm first, and log a stack trace so a stray trigger is visible in DevTools.
page=replace(page,'onClick={() => void cancelFace(activeFaceControl).catch((caught) => setError(String(caught)))}','onClick={() => { if (!window.confirm("確定要停止人臉精修嗎？原片會保留。")) return; console.warn("[H3] face stop clicked", new Error().stack); void cancelFace(activeFaceControl).catch((caught) => setError(String(caught))); }}')
page=replace(page,'畫質提升 · 720p · {video.enhancementStrength}%','畫質提升 · {Math.min(video.width ?? 0, video.height ?? 0) || 720}p · {video.enhancementStrength}%')
if not HAS_SINGULARITY:
    page=replace(page,'                      <option value="singularity-v2">Singularity V2 · INT8</option>\n','')
if not HAS_VOSR:
    page=replace(page,'<button role="menuitem" onClick={() => { setOpenMenu(null); onVOSR(video); }}>VOSR 精修</button>','<button disabled title="VOSR 未安裝" role="menuitem" onClick={() => { setOpenMenu(null); onVOSR(video); }}>VOSR 精修</button>')
# mk14: per-mode defaults (text: Singularity V2 when installed; image/reference: original H3).
TEXT_MODEL='singularity-v2' if HAS_SINGULARITY else 'original'
page=replace(page,'useState<H3Model>("original")','useState<H3Model>('+json.dumps(TEXT_MODEL)+')')
page=replace(page,'useState<GenerationOptions["profile"]>("fast")','useState<GenerationOptions["profile"]>("cooled-turbo-8")')
page=replace(page,'useState<H3AttentionMethod>("sage")','useState<H3AttentionMethod>("sol")')
page=replace(page,'useState<GenerationOptions["resolution"]>("safe")','useState<GenerationOptions["resolution"]>("p480")')
page=replace(page,'  const [duration, setDuration] = useState(5);\n','  const [duration, setDuration] = useState(10);\n')
page=replace(page,'  const [soundEnabled, setSoundEnabled] = useState(true);\n','  const [soundEnabled, setSoundEnabled] = useState(true);\n  const modeSettings = useRef<Partial<Record<SourceMode, { h3Model: H3Model; profile: GenerationOptions["profile"]; attentionMethod: H3AttentionMethod; resolution: GenerationOptions["resolution"]; duration: number; aspect: GenerationOptions["aspect"]; soundEnabled: boolean }>>>({});\n')
page=replace(page,"""    if (nextMode === "reference") {
      setProfile(h3Model === "singularity-v2" ? "cooled-turbo-4" : "cooled-turbo-8");
      setResolution("native");
      setAspect("16:9");
    } else {
      if (resolution === "native") setResolution("safe");
      if (aspect === "4:3" || aspect === "3:4") setAspect("16:9");
    }
""","""    // Colab: each mode keeps its own settings for this page; the first visit uses that mode's defaults.
    modeSettings.current[sourceMode] = { h3Model, profile, attentionMethod, resolution, duration, aspect, soundEnabled };
    const next = modeSettings.current[nextMode] ?? { h3Model: nextMode === "text" ? """+json.dumps(TEXT_MODEL)+""" as H3Model : "original" as H3Model, profile: "cooled-turbo-8" as const, attentionMethod: "sol" as const, resolution: "p480" as const, duration: 10, aspect: "16:9" as const, soundEnabled: true };
    setH3Model(next.h3Model);
    setProfile(next.profile);
    setAttentionMethod(next.attentionMethod);
    setResolution(next.resolution);
    setDuration(next.duration);
    setAspect(next.aspect);
    setSoundEnabled(next.soundEnabled);
""")
# mk25: all three video modes expose true native 12/16/20 alongside the Turbo settings.
# Existing QUALITY 20 is moved into the common options so reference mode gets it too.
page=replace(page,'                          <option value="quality">QUALITY · 原生 20 步</option>\n','')
page=replace(page,'                          <option value="cooled-turbo-8">{h3Model === "singularity-v2" ? "8步 · 實驗" : "8步"}</option>\n',
                  '                          <option value="cooled-turbo-8">{h3Model === "singularity-v2" ? "8步 · 實驗" : "8步"}</option>\n'
                  '                          <option value="quality-12">QUALITY · 原生 12 步</option>\n'
                  '                          <option value="quality-16">QUALITY · 原生 16 步</option>\n'
                  '                          <option value="quality">QUALITY · 原生 20 步</option>\n',2)
page=replace(page,'profile === "quality" ? 20 : profile === "cooled-turbo-8" ? 8 : 6;',
                  'profile === "quality-12" ? 12 : profile === "quality-16" ? 16 : profile === "quality" ? 20 : profile === "cooled-turbo-8" ? 8 : 6;')
page=replace(page,' · Turbo {referenceVideoSteps(profile)} 步</span>',
                  ' · {profile === "quality" || profile === "quality-12" || profile === "quality-16" ? "原生" : "Turbo"} {referenceVideoSteps(profile)} 步</span>')
page=replace(page,'{profile === "quality" ? "原生取樣，不使用加速 LoRA" : "自動搭配 LightX2V Ref2VA Turbo 4-step；6／8 步為實驗設定"}',
                  '{profile === "quality" || profile === "quality-12" || profile === "quality-16" ? "原生取樣，不使用加速 LoRA" : "自動搭配 LightX2V Ref2VA Turbo 4-step；6／8 步為實驗設定"}')
save('h3-local-studio/app/page.tsx',page)
# Once the face job has produced its result, finish compositing even if a stop flag was set late.
fr='h3-local-studio/lib/face-refine.ts'
s=pristine(fr)
s=replace(s,'    const raw = await sampleFace(source, input, control, onStatus, picks);\n    checkStop();\n','    const raw = await sampleFace(source, input, control, onStatus, picks);\n')
# mk32：「停止人臉精修」要真的中斷 GPU 上的精修。原本 cancelFace 在等 /queue 回應時，輪詢已先把 control.promptId 清掉，
# 結果送出 delete:[null]、沒有 interrupt，精修繼續在背景跑並卡住下一支影片。（repo main 已修；釘選的 commit 還沒有，所以這裡補。）
s=replace(s,'  control.stop = true;\n  if (!control.promptId) return;\n','  const promptId = control.promptId;\n  control.stop = true;\n  if (!promptId) return;\n')
s=replace(s,'item[1] === control.promptId)) {','item[1] === promptId)) {')
s=replace(s,'body: JSON.stringify({ prompt_id: control.promptId }) });','body: JSON.stringify({ prompt_id: promptId }) });')
s=replace(s,'body: JSON.stringify({ delete: [control.promptId] }) });','body: JSON.stringify({ delete: [promptId] }) });')
save(fr,s)
speech=pristine('h3-local-studio/app/speech-studio.tsx')
speech=replace(speech,'useState<"drama3" | "gemini">("drama3")','useState<"drama3" | "gemini">("gemini")')
speech=''.join(line for line in speech.splitlines(keepends=True) if not ('<button ' in line and 'setEngine("drama3")' in line) and '<div hidden={engine !== "drama3"}>' not in line)
save('h3-local-studio/app/speech-studio.tsx',speech)
if SWAP_AVAILABLE:
    swap=pristine('h3-local-studio/app/swap-studio.tsx')
    default=next(iter(SWAP_AVAILABLE))
    swap=replace(swap,'useState<SwapMode>("head")','useState<SwapMode>('+json.dumps(default)+')')
    swap=replace(swap,'useState(buildSwapPrompt("head"))','useState(buildSwapPrompt('+json.dumps(default)+'))')
    import re
    for mode in ('head','character','akatz-v1.5','untmods-face'):
        if mode not in SWAP_AVAILABLE:
            swap,n=re.subn(r'<option value="'+re.escape(mode)+r'">[^<]*</option>','',swap)
            assert n==1
    save('h3-local-studio/app/swap-studio.tsx',swap)
# Safe-long's CPU upscale no longer invokes the Windows Recycle Bin.
# Keep the source video; disabled deletion/continuation endpoints are blocked by nginx below.
api=pristine('integrations/ComfyUI-H3-Studio/h3_studio_api.py')
api=replace(api,'    recycle_file(source)\n','    # Colab: retain the local source; no Windows Recycle Bin call.\n')
# mk22: last ComfyUI log lines for the queue popup (progress bars collapsed to their latest update).
H3_LOG_ROUTE=r'''H3_COMFY_LOG = Path(__H3_LOG__)
_H3_ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")
_H3_PROGRESS = re.compile(r"\d+%\||\d+/\d+ \[")


@PromptServer.instance.routes.get("/h3-studio/log-tail")
async def h3_log_tail(request):
    try:
        count = max(1, min(40, int(request.query.get("lines", "10"))))
    except ValueError:
        count = 10
    try:
        with H3_COMFY_LOG.open("rb") as handle:
            handle.seek(0, os.SEEK_END)
            handle.seek(max(0, handle.tell() - 65536))
            text = handle.read().decode("utf-8", errors="replace")
    except OSError:
        return web.json_response({"lines": []}, headers={"Cache-Control": "no-store"})
    lines = []
    for raw in re.split(r"[\r\n]+", _H3_ANSI.sub("", text)):
        line = raw.rstrip()
        if not line.strip():
            continue
        if lines and _H3_PROGRESS.search(line) and _H3_PROGRESS.search(lines[-1]):
            lines[-1] = line[-240:]
        else:
            lines.append(line[-240:])
    return web.json_response({"lines": lines[-count:]}, headers={"Cache-Control": "no-store"})


@PromptServer.instance.routes.get("/h3-studio/outputs")
'''.replace('__H3_LOG__',repr(str(ROOT/'h3_comfy.log')))
api=replace(api,'@PromptServer.instance.routes.get("/h3-studio/outputs")\n',H3_LOG_ROUTE)
# mk30: accept the Colab-only native 12/16-step profiles in saved work metadata (otherwise the profile is
# dropped and 「再做一次」 shows 作品無原始設定).
api=replace(api,'{"fast", "cooled-fast", "cooled-turbo-4", "cooled-turbo-8", "quality", "safe-long", "low-vram"}',
                '{"fast", "cooled-fast", "cooled-turbo-4", "cooled-turbo-8", "quality-12", "quality-16", "quality", "safe-long", "low-vram"}')
save('integrations/ComfyUI-H3-Studio/h3_studio_api.py',api)
# Enhancement: selectable 720p / 1080p / 1440p target (short side).
enh='integrations/ComfyUI-H3-Studio/h3_studio_enhance.py'
s=pristine(enh)
s=replace(s,'''def target_dimensions(width, height):
    if min(width, height) >= 720:
        raise ValueError("目前提供升至 720p，請選擇短邊小於 720 的影片。")
    scale = 720 / min(width, height)''','''def target_dimensions(width, height, target=720):
    if target not in (720, 1080, 1440):
        raise ValueError("目標解析度只支援 720p、1080p、1440p。")
    if min(width, height) >= target:
        raise ValueError(f"來源短邊已達 {min(width, height)}，請選擇高於來源的目標解析度。")
    scale = target / min(width, height)''')
s=replace(s,'''        width, height = target_dimensions(info["width"], info["height"])
        if max(width, height) > 2160:
            raise ValueError("影片比例過長，720p 長邊上限為 2160。")''','''        target = int(options.get("target", 720))
        width, height = target_dimensions(info["width"], info["height"], target)
        if max(width, height) > target * 3:
            raise ValueError(f"影片比例過長，{target}p 長邊上限為 {target * 3}。")''')
save(enh,s)
lib='h3-local-studio/lib/enhance.ts'
s=pristine(lib)
s=replace(s,'export async function prepareEnhancement(source: File | GeneratedVideo, start: number, duration: number) {','export async function prepareEnhancement(source: File | GeneratedVideo, start: number, duration: number, target = 720) {')
s=replace(s,'    body.append("duration", String(duration));\n','    body.append("duration", String(duration));\n    body.append("target", String(target));\n')
s=replace(s,'JSON.stringify({ source, start, duration })','JSON.stringify({ source, start, duration, target })')
save(lib,s)
ui='h3-local-studio/app/enhance-studio.tsx'
s=pristine(ui)
s=replace(s,'  const [strength, setStrength] = useState(50);\n','  const [strength, setStrength] = useState(50);\n  const [target, setTarget] = useState(720);\n')
s=replace(s,'prepareEnhancement(source, segment ? start : 0, segment ? duration : 0)','prepareEnhancement(source, segment ? start : 0, segment ? duration : 0, target)')
s=replace(s,'<p>升至 720p，保留原比例','<p>升至所選解析度（短邊），保留原比例')
s=replace(s,'      <label className="enhancement-strength">','      <label><span>目標解析度</span><select value={target} disabled={locked} onChange={(event) => { reset(); setTarget(Number(event.target.value)); }}><option value={720}>720p</option><option value={1080}>1080p</option><option value={1440}>1440p</option></select></label>\n      <label className="enhancement-strength">')
s=replace(s,'<h3>畫質提升 · 720p · ','<h3>畫質提升 · {result.width} × {result.height} · ')
save(ui,s)
# Enhancement: no 15 s cap; duration auto-fills with the source video's full length.
s=(REPO/enh).read_text(encoding='utf-8')
s=replace(s,'''        if duration > 15.1:
            raise ValueError("第一版每次最多處理 15 秒，請改用片段測試。")
''','')
save(enh,s)
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'每次最多 15 秒；建議先測試 3 秒。','片段長度會自動帶入影片全長，可自行縮短先測試。')
s=replace(s,'aria-label="來源影片預覽" />','aria-label="來源影片預覽" onLoadedMetadata={(event) => { const total = event.currentTarget.duration; if (Number.isFinite(total) && total > 0) { setStart(0); setDuration(Math.floor(total * 10) / 10); } }} />')
s=replace(s,'min="0.1" max="15" step="0.1"','min="0.1" step="0.1"')
s=replace(s,' || duration > 15))}','))}')
save(ui,s)
# Enhancement joins the shared generation queue (no screen lock; can queue behind other jobs / several targets).
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'import { useEffect, useState } from "react";','import { useEffect, useRef, useState } from "react";')
s=replace(s,'function SeedVRStudio({ videos, connected, disabled, executionDisabled, onBusyChange, onComplete }: {','function SeedVRStudio({ videos, connected, disabled, executionDisabled, onBusyChange, onComplete, enqueue }: {')
s=replace(s,'  onComplete: () => void;\n}) {','  onComplete: () => void;\n  enqueue: (label: string, run: (onStatus: (message: string) => void) => Promise<void>, onDiscard?: () => void) => number;\n}) {\n  void executionDisabled; void onBusyChange;\n  const selection = useRef(0);')
s=replace(s,'  function reset() { setCache(null);','  function reset() { selection.current += 1; setCache(null);')
start=s.index('  async function enhance() {'); end=s.index('  const locked = busy || disabled;')
s=s[:start]+'''  function enhance() {
    if (!source) return;
    const chosenStrength = strength, src = source, seg = segment, st = start, du = duration, tg = target, token = selection.current;
    const sameSelection = () => token === selection.current;
    const name = src instanceof File ? src.name : src.filename;
    setError(""); setStatus("已加入生成佇列，可切換到其他功能。");
    enqueue(`畫質提升 · ${name} · ${tg}p · ${chosenStrength}%`, async (onStatus) => {
      const report = (text: string) => { if (sameSelection()) setStatus(text); onStatus(text); };
      let current = sameSelection() ? cache : null;
      try {
        if (!current) {
          report("正在準備影片片段…");
          current = { input: await prepareEnhancement(src, seg ? st : 0, seg ? du : 0, tg), raw: null };
          if (sameSelection()) setCache(current);
        }
        if (chosenStrength > 0 && !current.raw) {
          current = { ...current, raw: await runEnhancement(current.input, report) };
          if (sameSelection()) setCache(current);
        }
        report("正在混合原片並另存成品…");
        const completed = await blendEnhancement(current.input, current.raw, chosenStrength);
        if (sameSelection()) setResult(completed);
        onComplete();
        report(`已完成 ${completed.width} × ${completed.height} · ${chosenStrength}% 強度`);
      } catch (caught) {
        if (sameSelection()) setError(caught instanceof Error ? caught.message : "影片畫質提升失敗。");
        throw caught;
      }
    }, () => { if (sameSelection()) setStatus("已從佇列移除畫質提升。"); });
  }

'''+s[end:]
s=replace(s,'disabled={!connected || locked || executionDisabled || !source ||','disabled={!connected || locked || !source ||')
s=replace(s,'{busy ? `正在處理 · ${elapsed} 秒` : cache?.raw ? "套用強度並另存" : "開始提升畫質"}','{cache?.raw ? "套用強度並另存（加入佇列）" : "加入畫質提升佇列"}')
s=replace(s,'      {executionDisabled && <p>可先設定，等待生成佇列完成後即可開始提升畫質。</p>}\n','')
save(ui,s)
qp='h3-local-studio/app/generation-queue-panel.tsx'
s=pristine(qp)
s=replace(s,'job.label.startsWith("RTX · ") ? "RTX 提升" :','job.label.startsWith("RTX · ") ? "RTX 提升" : job.label.startsWith("畫質提升 · ") ? "畫質提升" : job.label.startsWith("VOSR 1× 精修 · ") ? "VOSR 精修" :')
# mk22: while the popup of a running job is open, show the last 10 ComfyUI log lines (refresh every 2 s).
s=replace(s,'import { KeyboardEvent, useRef, useState } from "react";','import { KeyboardEvent, useEffect, useRef, useState } from "react";')
s=replace(s,'  const toggle = useRef<HTMLButtonElement>(null);\n',r'''  const toggle = useRef<HTMLButtonElement>(null);
  const [logLines, setLogLines] = useState<string[]>([]);
  const logJobRunning = popup !== null && jobs.some((job) => job.id === popup.id && job.status === "running");
  useEffect(() => {
    if (!logJobRunning) { setLogLines([]); return; }
    let alive = true;
    const load = async () => {
      try {
        const response = await fetch("/comfy/h3-studio/log-tail?lines=10", { cache: "no-store" });
        if (response.ok && alive) setLogLines(((await response.json()) as { lines?: string[] }).lines ?? []);
      } catch { /* tunnel hiccup: keep the previous lines */ }
    };
    void load();
    const timer = window.setInterval(() => void load(), 2000);
    return () => { alive = false; window.clearInterval(timer); };
  }, [logJobRunning]);
''')
s=replace(s,'''                  <p className="queue-message" role={job.status === "failed" ? "alert" : "status"}>{job.message}</p>
''',r'''                  <p className="queue-message" role={job.status === "failed" ? "alert" : "status"}>{job.message}</p>
                  {job.status === "running" && logLines.length > 0 && (
                    <pre aria-label="ComfyUI 日誌" style={{ margin: "8px 0 0", padding: "8px 10px", borderRadius: 8, background: "#0e0d13", color: "#b9b3c6", fontSize: 11, lineHeight: 1.5, whiteSpace: "pre-wrap", overflowWrap: "anywhere", maxHeight: 220, overflowY: "auto" }}>{logLines.join("\n")}</pre>
                  )}
''')
save(qp,s)
# Enhancement auto-chunking: long clips are split into short segments (RAM stays bounded), then joined.
s=(REPO/enh).read_text(encoding='utf-8')
s=replace(s,'import math\n','import math\nimport re\n')
s=replace(s,'''            options = await request.json()
            item = options.get("source", {})
            if not isinstance(item, dict):
                raise ValueError("無效的影片來源。")
            source = resolve_output_video(item.get("filename"), item.get("subfolder", ""))
            source_name = source.name
''','''            options = await request.json()
            if options.get("fromSourceId"):
                base_clip, base_manifest = input_paths(options["fromSourceId"])
                source = base_clip
                source_name = json.loads(base_manifest.read_text(encoding="utf-8")).get("sourceName", base_clip.name) if base_manifest.is_file() else base_clip.name
            else:
                item = options.get("source", {})
                if not isinstance(item, dict):
                    raise ValueError("無效的影片來源。")
                source = resolve_output_video(item.get("filename"), item.get("subfolder", ""))
                source_name = source.name
''')
s=replace(s,'def blend_video(source_id, raw, strength):','def blend_video(source_id, raw, strength, segment=False):')
s=replace(s,'    output = Path(folder_paths.get_output_directory()) / "video" / f"H3_Studio_Enhanced_{uuid.uuid4().hex[:12]}.mp4"\n',
            '    output = (clip.parent / f"{source_id}_seg_{uuid.uuid4().hex[:12]}.mp4") if segment else Path(folder_paths.get_output_directory()) / "video" / f"H3_Studio_Enhanced_{uuid.uuid4().hex[:12]}.mp4"\n')
s=replace(s,'        metadata = {"kind": "video", "enhanced": True, "enhancementStrength": strength,\n                    "enhancementSource": info["sourceName"], "width": width, "height": height,',
            '        if segment:\n            return {"segmentFile": output.name}\n        metadata = {"kind": "video", "enhanced": True, "enhancementStrength": strength,\n                    "enhancementSource": info["sourceName"], "width": width, "height": height,')
s=replace(s,'blend_video, data.get("sourceId"), data.get("raw"), data.get("strength"))','blend_video, data.get("sourceId"), data.get("raw"), data.get("strength"), bool(data.get("segment")))')
s+='''

SEGMENT_NAME = re.compile(r"^[0-9a-f]{32}_seg_[0-9a-f]{12}\\.mp4$")


def join_segments(names, strength):
    root = Path(folder_paths.get_input_directory()) / "h3_enhance"
    if not isinstance(names, list) or not 2 <= len(names) <= 1000 or not all(isinstance(n, str) and SEGMENT_NAME.match(n) for n in names):
        raise ValueError("無效的分段清單。")
    if isinstance(strength, bool) or not isinstance(strength, (int, float)) or not 0 <= strength <= 100:
        raise ValueError("強度必須介於 0～100%。")
    files = [root / n for n in names]
    if not all(f.is_file() for f in files):
        raise ValueError("分段檔案不存在，請重新處理。")
    manifest = root / f"{names[0][:32]}.json"
    info = json.loads(manifest.read_text(encoding="utf-8")) if manifest.is_file() else {}
    executable = shutil.which("ffmpeg")
    if not executable:
        raise ValueError("找不到 FFmpeg。")
    output = Path(folder_paths.get_output_directory()) / "video" / f"H3_Studio_Enhanced_{uuid.uuid4().hex[:12]}.mp4"
    output.parent.mkdir(parents=True, exist_ok=True)
    listing = root / f"join_{uuid.uuid4().hex[:12]}.txt"
    listing.write_text("".join(f"file '{f.as_posix()}'\\n" for f in files), encoding="utf-8")
    try:
        result = subprocess.run([executable, "-hide_banner", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", str(listing),
                                 "-map", "0:v:0", "-map", "0:a?", "-c:v", "libx264", "-crf", "18", "-preset", "fast",
                                 "-c:a", "aac", "-y", str(output)], capture_output=True, text=True, timeout=7200)
        if result.returncode:
            raise ValueError(result.stderr.strip()[-1200:] or "分段接合失敗。")
        final = probe_video(output)
        metadata = {"kind": "video", "enhanced": True, "enhancementStrength": strength,
                    "enhancementSource": info.get("sourceName", "影片"), "width": final["width"], "height": final["height"],
                    "duration": final["duration"], "sound": final["sound"], "extendable": False, "enhancementSegments": len(files)}
        metadata_path_for(output).write_text(json.dumps(metadata, ensure_ascii=False), encoding="utf-8")
        for f in files:
            f.unlink(missing_ok=True)
        return {"filename": output.name, "subfolder": "video", "type": "output",
                "modifiedAt": output.stat().st_mtime, **metadata}
    except Exception:
        output.unlink(missing_ok=True)
        raise
    finally:
        listing.unlink(missing_ok=True)


@PromptServer.instance.routes.post("/h3-studio/enhance/join")
async def join_output(request):
    try:
        data = await request.json()
        result = await asyncio.to_thread(join_segments, data.get("segments"), data.get("strength"))
        return web.json_response(result)
    except (ValueError, TypeError, KeyError, OSError, subprocess.SubprocessError) as error:
        return web.json_response({"error": str(error)}, status=400)
'''
save(enh,s)
s=(REPO/lib).read_text(encoding='utf-8')
s+='''
export async function prepareEnhancementFrom(sourceId: string, start: number, duration: number, target = 720) {
  return await (await request("/h3-studio/enhance/prepare", {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ fromSourceId: sourceId, start, duration, target }),
  })).json() as EnhancementInput;
}

export async function blendEnhancementSegment(source: EnhancementInput, raw: GeneratedVideo | null, strength: number) {
  return await (await request("/h3-studio/enhance/blend", {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ sourceId: source.sourceId, raw, strength, segment: true }),
  })).json() as { segmentFile: string };
}

export async function joinEnhancement(segments: string[], strength: number) {
  return await (await request("/h3-studio/enhance/join", {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ segments, strength }),
  })).json() as GeneratedVideo;
}
'''
save(lib,s)
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'import { blendEnhancement, EnhancementInput, prepareEnhancement, runEnhancement } from "@/lib/enhance";','import { blendEnhancement, blendEnhancementSegment, EnhancementInput, joinEnhancement, prepareEnhancement, prepareEnhancementFrom, runEnhancement } from "@/lib/enhance";')
s=replace(s,'  const [target, setTarget] = useState(720);\n','  const [target, setTarget] = useState(720);\n  const [sourceLength, setSourceLength] = useState(0);\n')
s=replace(s,'if (Number.isFinite(total) && total > 0) { setStart(0);','if (Number.isFinite(total) && total > 0) { setSourceLength(Math.floor(total * 10) / 10); setStart(0);')
s=replace(s,'      let current = sameSelection() ? cache : null;\n      try {','''      const total = seg ? du : (sourceLength || du);
      const offset = seg ? st : 0;
      const chunk = tg >= 1440 ? 5 : tg >= 1080 ? 10 : 20;
      if (total > chunk + 0.05) {
        try {
          report("正在準備完整片段…");
          const base = await prepareEnhancement(src, offset, total, tg);
          const pieces: [number, number][] = [];
          for (let t = 0; t < total - 0.05; t += chunk) pieces.push([t, t + chunk >= total - 0.05 ? 0 : chunk]);
          const segments: string[] = [];
          for (const [index, [at, length]] of pieces.entries()) {
            const tag = `第 ${index + 1}/${pieces.length} 段`;
            report(`${tag}：準備中…`);
            const input = await prepareEnhancementFrom(base.sourceId, at, length, tg);
            const raw = chosenStrength > 0 ? await runEnhancement(input, (message) => report(`${tag}：${message}`)) : null;
            report(`${tag}：混合原片…`);
            segments.push((await blendEnhancementSegment(input, raw, chosenStrength)).segmentFile);
          }
          report("正在接合所有分段並另存成品…");
          const completed = await joinEnhancement(segments, chosenStrength);
          if (sameSelection()) setResult(completed);
          onComplete();
          report(`已完成 ${completed.width} × ${completed.height} · ${pieces.length} 段 · ${chosenStrength}% 強度`);
        } catch (caught) {
          if (sameSelection()) setError(caught instanceof Error ? caught.message : "影片畫質提升失敗。");
          throw caught;
        }
        return;
      }
      let current = sameSelection() ? cache : null;
      try {''')
s=replace(s,'片段長度會自動帶入影片全長，可自行縮短先測試。','片段長度會自動帶入影片全長；長片會自動分段處理（720p 每段 20 秒、1080p 10 秒、1440p 5 秒）再接合。')
save(ui,s)
# LongMedia @96c50f7 wraps H3 FinalLayer.forward with the old 4-arg signature; ComfyUI v0.37.0
# passes 7 args (sigma, sample_sigmas, shifts) -> Ref2VA fails "_streamed_forward() takes 5 positional
# arguments but 8 were given". Delegate to the stock forward when the new args are present.
lm_dir=COMFY/'custom_nodes/ComfyUI-MiniMax-H3-LongMedia'
lm_src=subprocess.check_output(['git','-C',str(lm_dir),'show','96c50f7febc8c2e80c2a50d030d6cb3ff3de7ee3:nodes.py'],text=True,encoding='utf-8')
lm_src=replace(lm_src,'        def _streamed_forward(layer, x, t_emb, video_seg, audio_seg):\n',
    '        def _streamed_forward(layer, x, t_emb, video_seg, audio_seg, *extra):\n'
    '            if extra:\n'
    '                return original(x, t_emb, video_seg, audio_seg, *extra)\n')
(lm_dir/'nodes.py').write_text(lm_src,encoding='utf-8')
# Enhancement speed: VAE tiles / SeedVR2 temporal chunk scale with VRAM; RAM-aware segment length; join copies video.
s=(REPO/lib).read_text(encoding='utf-8')
s=replace(s,'export function buildEnhancementWorkflow(source: EnhancementInput) {','export type EnhancementTuning = { tile: number; temporal: number; chunkFrames: number };\nconst DEFAULT_TUNING: EnhancementTuning = { tile: 512, temporal: 16, chunkFrames: 9 };\n\nexport function buildEnhancementWorkflow(source: EnhancementInput, tuning: EnhancementTuning = DEFAULT_TUNING) {')
s=replace(s,'"7": node("VAEEncodeTiled", { pixels: ["4", 0], vae: ["5", 0], tile_size: 512, overlap: 128, temporal_size: 16, temporal_overlap: 8 }),','"7": node("VAEEncodeTiled", { pixels: ["4", 0], vae: ["5", 0], tile_size: tuning.tile, overlap: 128, temporal_size: tuning.temporal, temporal_overlap: 8 }),')
s=replace(s,'"chunking_mode.frames_per_chunk": 9 }),','"chunking_mode.frames_per_chunk": tuning.chunkFrames }),')
s=replace(s,'"12": node("VAEDecodeTiled", { samples: ["11", 0], vae: ["5", 0], tile_size: 512, overlap: 128, temporal_size: 16, temporal_overlap: 8 }),','"12": node("VAEDecodeTiled", { samples: ["11", 0], vae: ["5", 0], tile_size: tuning.tile, overlap: 128, temporal_size: tuning.temporal, temporal_overlap: 8 }),')
s=replace(s,'''  onStatus("SeedVR2 正在載入模型與提升畫質…");
  const result = await (await request("/prompt", {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ prompt: buildEnhancementWorkflow(source), client_id: crypto.randomUUID() }),''','''  onStatus("SeedVR2 正在載入模型與提升畫質…");
  const tuning = await enhancementTuning();
  const result = await (await request("/prompt", {
    method: "POST", headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ prompt: buildEnhancementWorkflow(source, tuning), client_id: crypto.randomUUID() }),''')
s+='''
/** Bigger VAE tiles / temporal chunks on large GPUs: far fewer passes (defaults were sized for 12 GB laptops). */
export async function enhancementTuning(): Promise<EnhancementTuning> {
  try {
    const stats = await (await request("/system_stats")).json() as { devices?: { vram_total?: number }[] };
    const vram = (stats.devices?.[0]?.vram_total ?? 0) / 1024 ** 3;
    if (vram >= 38) return { tile: 1024, temporal: 32, chunkFrames: 21 };
    if (vram >= 20) return { tile: 768, temporal: 24, chunkFrames: 13 };
  } catch { /* fall back to the original conservative settings */ }
  return DEFAULT_TUNING;
}

/** System RAM in GiB, used to size segments (frames are held in RAM several times over). */
export async function systemRamGiB() {
  try {
    const stats = await (await request("/system_stats")).json() as { system?: { ram_total?: number } };
    return (stats.system?.ram_total ?? 0) / 1024 ** 3;
  } catch { return 0; }
}
'''
save(lib,s)
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'joinEnhancement, prepareEnhancement, prepareEnhancementFrom, runEnhancement } from "@/lib/enhance";','joinEnhancement, prepareEnhancement, prepareEnhancementFrom, runEnhancement, systemRamGiB } from "@/lib/enhance";')
s=replace(s,'      const chunk = tg >= 1440 ? 5 : tg >= 1080 ? 10 : 20;\n','      const ram = await systemRamGiB();\n      const chunk = tg >= 1440 ? (ram >= 100 ? 5 : 3) : tg >= 1080 ? (ram >= 60 ? 10 : 6) : 20;\n')
s=replace(s,'（720p 每段 20 秒、1080p 10 秒、1440p 5 秒）','（依機器記憶體自動決定每段長度）')
save(ui,s)
s=(REPO/enh).read_text(encoding='utf-8')
s=replace(s,'''"-map", "0:v:0", "-map", "0:a?", "-c:v", "libx264", "-crf", "18", "-preset", "fast",
                                 "-c:a", "aac", "-y", str(output)]''','''"-map", "0:v:0", "-map", "0:a?", "-c:v", "copy",
                                 "-c:a", "aac", "-y", str(output)]''')
save(enh,s)
# Enhancement speed 2: bicubic pre-upscale (PIL lanczos was single-core per frame) and 64px VAE tile overlap on big tiles.
s=(REPO/lib).read_text(encoding='utf-8')
s=replace(s,'"resize_type.crop": "disabled", scale_method: "lanczos" }),','"resize_type.crop": "disabled", scale_method: "bicubic" }),')
s=replace(s,'export type EnhancementTuning = { tile: number; temporal: number; chunkFrames: number };\nconst DEFAULT_TUNING: EnhancementTuning = { tile: 512, temporal: 16, chunkFrames: 9 };','export type EnhancementTuning = { tile: number; overlap: number; temporal: number; chunkFrames: number };\nconst DEFAULT_TUNING: EnhancementTuning = { tile: 512, overlap: 128, temporal: 16, chunkFrames: 9 };')
s=replace(s,'tile_size: tuning.tile, overlap: 128, temporal_size: tuning.temporal','tile_size: tuning.tile, overlap: tuning.overlap, temporal_size: tuning.temporal',2)
s=replace(s,'return { tile: 1024, temporal: 32, chunkFrames: 21 };','return { tile: 1024, overlap: 64, temporal: 32, chunkFrames: 21 };')
s=replace(s,'return { tile: 768, temporal: 24, chunkFrames: 13 };','return { tile: 768, overlap: 64, temporal: 24, chunkFrames: 13 };')
# Total processing time is saved with the result (shown on the card as 耗時 N 秒).
s=replace(s,'export async function blendEnhancement(source: EnhancementInput, raw: GeneratedVideo | null, strength: number) {','export async function blendEnhancement(source: EnhancementInput, raw: GeneratedVideo | null, strength: number, elapsed?: number) {')
s=replace(s,'    body: JSON.stringify({ sourceId: source.sourceId, raw, strength }),','    body: JSON.stringify({ sourceId: source.sourceId, raw, strength, elapsed }),')
s=replace(s,'export async function joinEnhancement(segments: string[], strength: number) {','export async function joinEnhancement(segments: string[], strength: number, elapsed?: number) {')
s=replace(s,'    body: JSON.stringify({ segments, strength }),','    body: JSON.stringify({ segments, strength, elapsed }),')
save(lib,s)
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'      const report = (text: string) => { if (sameSelection()) setStatus(text); onStatus(text); };\n','      const report = (text: string) => { if (sameSelection()) setStatus(text); onStatus(text); };\n      const began = Date.now();\n      const seconds = () => Math.max(1, Math.round((Date.now() - began) / 1000));\n')
s=replace(s,'const completed = await joinEnhancement(segments, chosenStrength);','const completed = await joinEnhancement(segments, chosenStrength, seconds());')
s=replace(s,'const completed = await blendEnhancement(current.input, current.raw, chosenStrength);','const completed = await blendEnhancement(current.input, current.raw, chosenStrength, seconds());')
save(ui,s)
s=(REPO/enh).read_text(encoding='utf-8')
s=replace(s,'def blend_video(source_id, raw, strength, segment=False):','def _seconds(elapsed):\n    return {"generationSeconds": max(1, round(float(elapsed)))} if isinstance(elapsed, (int, float)) and not isinstance(elapsed, bool) and math.isfinite(elapsed) and elapsed > 0 else {}\n\n\ndef blend_video(source_id, raw, strength, segment=False, elapsed=None):')
s=replace(s,'"duration": info["duration"], "sound": info["sound"], "extendable": False}','"duration": info["duration"], "sound": info["sound"], "extendable": False, **_seconds(elapsed)}')
s=replace(s,'bool(data.get("segment")))','bool(data.get("segment")), data.get("elapsed"))')
s=replace(s,'def join_segments(names, strength):','def join_segments(names, strength, elapsed=None):')
s=replace(s,'"extendable": False, "enhancementSegments": len(files)}','"extendable": False, "enhancementSegments": len(files), **_seconds(elapsed)}')
s=replace(s,'join_segments, data.get("segments"), data.get("strength"))','join_segments, data.get("segments"), data.get("strength"), data.get("elapsed"))')
save(enh,s)
# Segment length from a RAM budget (measured on Colab: 1440p x 3 s fit in 83 GB, 1080p x 10 s was OOM-killed at ~85 GB).
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'      const chunk = tg >= 1440 ? (ram >= 100 ? 5 : 3) : tg >= 1080 ? (ram >= 60 ? 10 : 6) : 20;\n','''      // ~0.25 GB of RAM per million output pixel-frames, plus ~15 GB for the loaded models; assume 16:9 at 30 fps.
      const budget = Math.max(20, (ram || 50) * 0.85 - 15) / 0.25 * 1e6;
      const chunk = Math.max(1, Math.min(20, Math.floor(budget / (tg * tg * 1.78 * 30) * 10) / 10));
''')
save(ui,s)
# Clear ComfyUI's node-output cache after every SeedVR2 job: each segment has new inputs, so old multi-GB frame tensors
# were kept in RAM and piled up across segments/jobs (1440p then 1080p was OOM-killed at ~85 GB).
s=(REPO/lib).read_text(encoding='utf-8')
s=replace(s,'''  onStatus("SeedVR2 正在載入模型與提升畫質…");
  const tuning = await enhancementTuning();''','''  onStatus("SeedVR2 正在載入模型與提升畫質…");
  // ComfyUI applies /free between queue items, checking at most every ~10 s; let a pending free land first.
  const wait = lastFree + 11000 - Date.now();
  if (wait > 0) await new Promise((resolve) => setTimeout(resolve, wait));
  const tuning = await enhancementTuning();''')
s=replace(s,'''  return waitForResult(result.prompt_id, () => onStatus("SeedVR2 正在提升畫質與 LAB 校色…"));''','''  try {
    return await waitForResult(result.prompt_id, () => onStatus("SeedVR2 正在提升畫質與 LAB 校色…"));
  } finally {
    await request("/free", { method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify({ free_memory: true }) }).catch(() => undefined);
    lastFree = Date.now();
  }''')
s=replace(s,'export type EnhancementTuning =','let lastFree = 0;\n\nexport type EnhancementTuning =')
save(lib,s)
# Optional FlashVSR engine for 畫質提升 (only when the FlashVSR node was installed in cell 6).
if globals().get('HAS_FLASHVSR'):
    s=(REPO/lib).read_text(encoding='utf-8')
    s+='''
export type EnhancementEngine = "seedvr2" | "flashvsr";

/** FlashVSR is trained for 4x: shrink to a quarter of the target (multiple of 32, so its 128-px output grid needs no crop),
 * upscale 4x, then resize to the exact target so blending/segment joins work unchanged. */
export function buildFlashWorkflow(source: EnhancementInput, vramGiB: number) {
  const node = (class_type: string, inputs: Record<string, unknown>) => ({ class_type, inputs });
  const quarter = (n: number) => Math.max(32, Math.round(n / 4 / 32) * 32);
  return {
    "1": node("LoadVideo", { file: source.inputFile }),
    "2": node("GetVideoComponents", { video: ["1", 0] }),
    "3": node("ResizeImageMaskNode", { input: ["2", 0], resize_type: "scale dimensions", "resize_type.width": quarter(source.outputWidth), "resize_type.height": quarter(source.outputHeight), "resize_type.crop": "disabled", scale_method: "area" }),
    "4": node("FlashVSRNode", { frames: ["3", 0], model: "FlashVSR-v1.1", mode: "tiny", scale: 4, tiled_vae: true, tiled_dit: vramGiB < 70, unload_dit: false, seed: 0 }),
    "5": node("ResizeImageMaskNode", { input: ["4", 0], resize_type: "scale dimensions", "resize_type.width": source.outputWidth, "resize_type.height": source.outputHeight, "resize_type.crop": "disabled", scale_method: "bicubic" }),
    "6": node("CreateVideo", { images: ["5", 0], fps: ["2", 2], bit_depth: 8 }),
    "7": node("SaveVideo", { video: ["6", 0], filename_prefix: `h3_enhance_cache/${source.sourceId}`, format: "mp4", "format.codec": "h264" }),
  };
}

async function vramGiB() {
  try {
    const stats = await (await request("/system_stats")).json() as { devices?: { vram_total?: number }[] };
    return (stats.devices?.[0]?.vram_total ?? 0) / 1024 ** 3;
  } catch { return 0; }
}
'''
    s=replace(s,'export async function runEnhancement(source: EnhancementInput, onStatus: (message: string) => void) {','export async function runEnhancement(source: EnhancementInput, onStatus: (message: string) => void, engine: EnhancementEngine = "seedvr2") {')
    s=replace(s,'  onStatus("SeedVR2 正在載入模型與提升畫質…");\n','  onStatus(engine === "flashvsr" ? "FlashVSR 正在載入模型與提升畫質（第一次會先下載模型）…" : "SeedVR2 正在載入模型與提升畫質…");\n')
    s=replace(s,'    body: JSON.stringify({ prompt: buildEnhancementWorkflow(source, tuning), client_id: crypto.randomUUID() }),','    body: JSON.stringify({ prompt: engine === "flashvsr" ? buildFlashWorkflow(source, await vramGiB()) : buildEnhancementWorkflow(source, tuning), client_id: crypto.randomUUID() }),')
    s=replace(s,'  if (!result.prompt_id) throw new Error(`SeedVR2 工作流無法執行：','  if (!result.prompt_id) throw new Error(`${engine === "flashvsr" ? "FlashVSR" : "SeedVR2"} 工作流無法執行：')
    s=replace(s,'    return await waitForResult(result.prompt_id, () => onStatus("SeedVR2 正在提升畫質與 LAB 校色…"));','    return await waitForResult(result.prompt_id, () => onStatus(engine === "flashvsr" ? "FlashVSR 正在提升畫質…" : "SeedVR2 正在提升畫質與 LAB 校色…"));')
    save(lib,s)
    s=(REPO/ui).read_text(encoding='utf-8')
    s=replace(s,'runEnhancement, systemRamGiB } from "@/lib/enhance";','runEnhancement, systemRamGiB, EnhancementEngine } from "@/lib/enhance";')
    s=replace(s,'function SeedVRStudio({ videos, connected, disabled, executionDisabled, onBusyChange, onComplete, enqueue }: {','function SeedVRStudio({ videos, connected, disabled, executionDisabled, onBusyChange, onComplete, enqueue, engine = "seedvr2" }: {')
    s=replace(s,'  enqueue: (label: string, run: (onStatus: (message: string) => void) => Promise<void>, onDiscard?: () => void) => number;\n}) {','  enqueue: (label: string, run: (onStatus: (message: string) => void) => Promise<void>, onDiscard?: () => void) => number;\n  engine?: EnhancementEngine;\n}) {')
    s=replace(s,'const chosenStrength = strength, src = source, seg = segment, st = start, du = duration, tg = target, token = selection.current;','const chosenStrength = strength, src = source, seg = segment, st = start, du = duration, tg = target, eng = engine, token = selection.current;')
    s=replace(s,'enqueue(`畫質提升 · ${name} · ${tg}p · ${chosenStrength}%`','enqueue(`畫質提升 · ${name} · ${tg}p · ${eng === "flashvsr" ? "FlashVSR" : "SeedVR2"} · ${chosenStrength}%`')
    s=replace(s,'const raw = chosenStrength > 0 ? await runEnhancement(input, (message) => report(`${tag}：${message}`)) : null;','const raw = chosenStrength > 0 ? await runEnhancement(input, (message) => report(`${tag}：${message}`), eng) : null;')
    s=replace(s,'current = { ...current, raw: await runEnhancement(current.input, report) };','current = { ...current, raw: await runEnhancement(current.input, report, eng) };')
    s=replace(s,'<option value="seedvr2">SeedVR2 · 提升至 720p</option>','<option value="seedvr2">SeedVR2 · 提升解析度（細節最多，較慢）</option><option value="flashvsr">FlashVSR · 提升解析度（快很多，畫面較平滑）</option>')
    s=replace(s,'<SeedVRStudio {...props}','<SeedVRStudio {...props} engine={method === "flashvsr" ? "flashvsr" : "seedvr2"}')
    save(ui,s)
# mk13: one 畫質處理方式 menu. Remount the SeedVR2 panel per method so a cached SeedVR2 result is never reused for FlashVSR.
s=(REPO/ui).read_text(encoding='utf-8')
s=replace(s,'<SeedVRStudio {...props}','<SeedVRStudio key={method} {...props}')
if not globals().get('HAS_FLASHVSR'):
    s=replace(s,'<option value="seedvr2">SeedVR2 · 提升至 720p</option>','<option value="seedvr2">SeedVR2 · 提升解析度</option>')
if not HAS_VOSR:
    s=replace(s,'<option value="vosr">VOSR · 1× 精修（維持原尺寸）</option>','')
if not HAS_SEED:
    s=replace(s,'useState(props.initialWork ? "vosr" : "seedvr2")','useState("vosr")')
    s,n=re.subn(r'<option value="(seedvr2|flashvsr)">[^<]*</option>','',s); assert n>=1
save(ui,s)
# VOSR: the worker runs with ComfyUI's own interpreter (Colab venv), not the Windows .venv path.
vosr='integrations/ComfyUI-H3-Studio/h3_studio_vosr.py'
s=pristine(vosr)
s=replace(s,'import subprocess\n','import subprocess\nimport sys\n')
s=replace(s,'root.parent / ".venv/Scripts/python.exe"','Path(sys.executable)',2)
# mk29: while the background downloader is fetching VOSR weights, report ready so the Studio can queue
# the job; the ComfyUI prompt gate (cell 5) holds it until the download completes.
s=replace(s,'    return {"ready": not missing, "missing": missing}','''    if missing and _bg_downloading("vosr"):
        return {"ready": True, "missing": [], "downloading": missing}
    return {"ready": not missing, "missing": missing}''')
s=replace(s,'def installation_status():','''def _bg_downloading(name):
    try:
        if not Path(f"/content/h3_bg_models/{name}.pending").exists():
            return False
        saved = json.loads(Path("/content/h3_bg_models.pid.json").read_text())
        stat = Path(f"/proc/{saved['pid']}/stat").read_text().rsplit(")", 1)[1].split()
        return stat[0] != "Z" and stat[19] == saved["token"]
    except Exception:
        return False


def installation_status():''')
save(vosr,s)
# Swap node: Colab runs without per-step cooldown, so accept 0 seconds.
sw='integrations/ComfyUI-H3-Studio/h3_studio_swap.py'
s=pristine(sw)
s=replace(s,'not 5 <= data["cooldownSeconds"] <= 600','not 0 <= data["cooldownSeconds"] <= 600')
s=replace(s,'每步降溫必須介於 5～600 秒。','每步降溫必須介於 0～600 秒。')
save(sw,s)
# mk19: head/character swap and face refine also load the Ref2VA main model; follow REF2VA_QUANT there too.
for rel in ('lib/swap.ts','lib/face-refine.ts'):
    p=WEB/rel; p.write_text(drop_sage(p.read_text(encoding='utf-8').replace('minimax_h3_ref2va_pruned_w4a8_mixed.safetensors',REF2VA_FILE),rel,1),encoding='utf-8')
# Ref2VA: make sure no other Studio source still hardcodes the W4A8 filename.
if REF2VA_FILE!='minimax_h3_ref2va_pruned_w4a8_mixed.safetensors':
    leftovers=[str(p.relative_to(WEB)) for d in ('app','lib','components') if (WEB/d).is_dir() for p in (WEB/d).rglob('*') if p.suffix in ('.ts','.tsx','.js','.json') and 'minimax_h3_ref2va_pruned_w4a8_mixed.safetensors' in p.read_text(encoding='utf-8',errors='ignore')]
    assert not leftovers, '以下檔案仍寫死 W4A8 Ref2VA 檔名，需要補丁：'+', '.join(leftovers)
print('Ref2VA 工作流使用：',REF2VA_FILE)
# Build before launching; catches source or Linux dependency failures early.
run(['npm','run','build'],cwd=WEB)
print('Colab UI patched; original generation graphs and queue retained.')


In [ ]:
# 8. 啟動原版服務，必須 HTTP 200 才算啟動成功。
import requests, requests.utils
# vinext production replies with zstd; urllib3 on Colab Py3.13 fails decoding chunked zstd. Only ask for gzip.
requests.utils.DEFAULT_ACCEPT_ENCODING='gzip, deflate'

DETACH="import subprocess,sys,json,os;p=subprocess.Popen(json.loads(sys.argv[1]),cwd=sys.argv[2],stdout=open(sys.argv[3],'w'),stderr=subprocess.STDOUT,stdin=subprocess.DEVNULL,start_new_session=True);print(p.pid,flush=True);os._exit(0)"
def launch_once(name,command,cwd,log,env=None):
    record=ROOT/(name+'.pid.json')
    if record.exists():
        saved=json.loads(record.read_text())
        if process_token(saved['pid'])==saved['token']:
            raise RuntimeError(name+' 已啟動；請先執行停止服務格，再重跑安裝／啟動')
        record.unlink()
    # Fully detached (own session, re-parented to init): interrupting/stopping a notebook cell
    # sends SIGTERM to the kernel's child processes, which previously took every service down.
    out=subprocess.run([sys.executable,'-c',DETACH,json.dumps([str(x) for x in command]),str(cwd),str(log)],capture_output=True,text=True,env=env,check=True,start_new_session=True)
    pid=int(out.stdout.strip()); time.sleep(0.3)
    token=process_token(pid)
    assert token is not None, name+' 立即退出，請查看 '+str(log)
    record.write_text(json.dumps({'pid':pid,'token':token}),encoding='utf-8')
    print('Started',name,pid)

def wait_url(url,seconds=300):
    until=time.monotonic()+seconds
    while time.monotonic()<until:
        try:
            r=requests.get(url,timeout=60)  # first SSR render can be slow
            if r.status_code==200: return r
        except requests.RequestException: pass
        time.sleep(2)
    raise RuntimeError('服務未就緒（只接受 HTTP 200）：'+url+'；查看下一格日誌')

# mk29：FlashVSR／VOSR 背景下載（第 6 格排程）；與服務並行，用到時 ComfyUI 會等它完成。
if any((ROOT/'h3_bg_models').glob('*.pending')):
    launch_once('h3_bg_models',[PYTHON,'-u',ROOT/'h3_bg_models.py',ROOT/'h3_bg_models.json'],ROOT,ROOT/'h3_bg_models.log')
service_env=os.environ.copy()
service_env['GEMINI_TTS_API_KEY']=GEMINI_KEY
launch_once('h3_comfy',[PYTHON,'main.py','--listen','127.0.0.1','--port','8188','--enable-cors-header','--disable-auto-launch','--extra-model-paths-config','extra_model_paths.yaml','--output-directory',OUTPUTS_ROOT],COMFY,ROOT/'h3_comfy.log',service_env)
stats=wait_url('http://127.0.0.1:8188/system_stats').json()
assert 'system' in stats and 'devices' in stats, '不是預期的 ComfyUI 服務'
# Production server (built in cell 7). No Vite dev server: avoids /@id/ 403, Host/Origin checks, slow on-demand compile.
launch_once('h3_studio',['npm','run','start','--','--hostname','127.0.0.1','--port','3000'],WEB,ROOT/'h3_studio.log')
wait_url('http://127.0.0.1:3000')
print('ComfyUI 與 Studio 已就緒')


In [ ]:
# 9. 檢查完整生成所需節點、模型、修臉狀態。缺少時不開公開網址。
base='http://127.0.0.1:8188'
r=requests.get(base+'/object_info',timeout=60); r.raise_for_status(); info=r.json()
required=['UNETLoader','CLIPLoader','VAELoader','MiniMaxH3TurboLoRA','MiniMaxH3TurboSampler','H3CooledTurboSampler','MiniMaxH3ReferenceToVideo','MiniMaxH3LatentLabLongMediaSetup','MiniMaxH3LatentLabLongMediaSampler','MiniMaxH3LatentLabLongMediaDecode','BlockSparseAttention','ModelAttentionBackend','SaveVideo','CreateVideo']
# Check every node class emitted by the two H3 graph builders, including cold-step/safe-long helpers.
source=(WEB/'lib/comfy.ts').read_text(encoding='utf-8')
for class_type in re.findall(r'node\("([A-Za-z0-9_]+)"',source):
    if class_type not in required: required.append(class_type)
missing=[name for name in required if name not in info]
assert not missing, '缺少生成節點：'+', '.join(missing)
unets=json.dumps(info['UNETLoader']['input']['required']['unet_name'])
assert REF2VA_FILE in unets, 'ComfyUI 看不到 Ref2VA 主模型：'+REF2VA_FILE
backend=info['ModelAttentionBackend']['input']['required']['attention']
assert 'comfy kitchen attention' in json.dumps(backend), 'Comfy Kitchen attention 未提供'
if INSTALL_FACE_REFINE:
    for query in ('','?mode=turbo-8&select=all_faces'):
        r=requests.get(base+'/h3-studio/face/status'+query,timeout=30); r.raise_for_status()
        ready=r.json(); assert ready.get('ready'), ready
if globals().get('HAS_FLASHVSR'): assert 'FlashVSRNode' in info, 'FlashVSR 節點未載入，請看 h3_comfy.log'
if globals().get('HAS_SINGULARITY'):
    assert SINGULARITY_MODEL in unets, 'ComfyUI 看不到 Singularity V2 主模型'
    assert SINGULARITY_LORA in json.dumps(info['LoraLoaderModelOnly']['input']['required']['lora_name']), 'ComfyUI 看不到 Singularity V2 加速 LoRA'
if globals().get('HAS_VOSR'):
    assert 'H3StudioVOSR' in info, 'H3StudioVOSR 節點未載入，請看 h3_comfy.log'
    r=requests.get(base+'/h3-studio/vosr/status',timeout=30); r.raise_for_status()
    assert r.json().get('ready'), ('VOSR 未就緒',r.json())
r=requests.get(base+'/h3-studio/outputs',timeout=30); r.raise_for_status()
_pending=[p.stem for p in (ROOT/'h3_bg_models').glob('*.pending')]
if _pending: print('背景下載中：',', '.join(_pending),'— 下載完成前使用會在佇列中等待。',flush=True)
print('節點與就緒狀態檢查通過；這尚未代替完整影片生成測試。')


In [ ]:
# 9b. mk28：環境快照存到 Drive（背景執行，不擋住後面各格；只在沒有可用快照時才存）。
# 存檔期間可以正常使用 Studio；/content/h3_snapshot.log 出現 DONE 前請勿刪除執行階段。
SNAPSHOT_SCRIPT=r"""
import os, shutil, subprocess, sys, time
from pathlib import Path
dest=Path(sys.argv[1]); local=Path(sys.argv[2]); paths=sys.argv[3:]
t=time.monotonic()
free=shutil.disk_usage(local.parent).free
if free<20*1024**3:
    sys.exit(f'[snapshot] SSD 剩餘 {free/1024**3:.1f} GiB，不足 20 GiB，略過快照')
print('[snapshot] 打包中（nice 降低優先序，不影響生成）…',flush=True)
excludes=['--exclude=__pycache__','--exclude=content/ComfyUI/output','--exclude=content/ComfyUI/input',
          '--exclude=content/ComfyUI/temp','--exclude=content/ComfyUI/user']
tar=subprocess.Popen(['nice','-n','15','tar','-C','/','--warning=no-file-changed',*excludes,'-cf','-',*paths],stdout=subprocess.PIPE)
z=subprocess.run(['nice','-n','15','zstd','-T0','-3','-q','-f','-o',str(local)],stdin=tar.stdout)
tar.stdout.close(); rc=tar.wait()
if rc not in (0,1) or z.returncode:
    local.unlink(missing_ok=True)
    sys.exit(f'[snapshot] 打包失敗 tar={rc} zstd={z.returncode}')
size=local.stat().st_size
print(f'[snapshot] 打包完成 {size/1024**3:.2f} GiB（{time.monotonic()-t:.0f} 秒），複製到 Drive…',flush=True)
dest.parent.mkdir(parents=True,exist_ok=True)
part=dest.with_name(dest.name+'.part')
try:
    shutil.copyfile(local,part)
    if part.stat().st_size!=size: raise OSError('Drive 複製大小不符')
    os.replace(part,dest)
    dest.with_name(dest.name+'.ok').write_text(f'{size}\n')
finally:
    part.unlink(missing_ok=True)
    local.unlink(missing_ok=True)
for old in dest.parent.glob('h3_env_*'):
    if old.name not in (dest.name,dest.name+'.ok'):
        old.unlink(missing_ok=True); print('[snapshot] 刪除舊快照',old.name,flush=True)
print(f'[snapshot] DONE {dest.name}（{size/1024**3:.2f} GiB），總計 {time.monotonic()-t:.0f} 秒',flush=True)
"""
if not USE_ENV_SNAPSHOT:
    print('USE_ENV_SNAPSHOT=False，不存快照。')
elif ENV_FROM_SNAPSHOT:
    print('本次環境來自快照，不需重存：',SNAP_FILE.name)
elif snapshot_valid() and not REBUILD_ENV_SNAPSHOT:
    print('Drive 已有可用快照：',SNAP_FILE.name)
else:
    try:
        uv_py=Path(subprocess.check_output([sys.executable,'-m','uv','python','dir'],text=True).strip())
    except Exception:
        uv_py=Path(os.path.realpath(PYTHON)).parents[1].parent
    assert uv_py.is_dir() and Path(os.path.realpath(PYTHON)).is_relative_to(uv_py.resolve()), ('找不到 uv Python 目錄',uv_py)
    paths=[p.relative_to('/').as_posix() for p in (VENV,COMFY,REPO,uv_py)]
    script=ROOT/'h3_snapshot.py'; script.write_text(SNAPSHOT_SCRIPT,encoding='utf-8')
    launch_once('h3_snapshot',[PYTHON,'-u',script,SNAP_FILE,ROOT/'h3_env_snapshot.tar.zst',*paths],ROOT,ROOT/'h3_snapshot.log')
    print('環境快照在背景存檔中 → ',SNAP_FILE)
    print('進度：/content/h3_snapshot.log（11b 也會顯示狀態）；出現 DONE 前請勿刪除執行階段。')


In [ ]:
# 10. 成功的影片與圖片（Qwen 生圖等）自動複製到 Drive；原檔留在 /content。
# 不會複製失敗工作的未完成檔案；複製失敗會每 5 秒重試。
copy_script=r"""

from pathlib import Path
import json, os, shutil, time, urllib.request

SOURCE=Path('/content/h3_outputs').resolve()
DEST=Path('/content/drive/MyDrive/H3_Colab/outputs')
VIDEO_EXT={'.mp4','.webm','.mov','.png','.jpg','.jpeg','.webp'}  # mk21: images too
copied={}
RUN_TAG=time.strftime('%Y%m%d-%H%M%S')  # mk21: new runtimes restart Comfy's counter at 00001

def same_file(a,b):
    try:
        x,y=a.stat(),b.stat()
        return x.st_size==y.st_size and abs(x.st_mtime-y.st_mtime)<2
    except OSError:
        return False

def drive_target(video):
    # Drive path for this output; never overwrite a different earlier file with the same name.
    base=DEST/video.relative_to(SOURCE)
    if not base.exists() or same_file(video,base):
        return base
    for earlier in base.parent.glob(f'{base.stem}_*{base.suffix}'):
        if same_file(video,earlier):
            return earlier  # already renamed by an earlier run of this copier
    n=0
    while True:
        tag=RUN_TAG if n==0 else f'{RUN_TAG}-{n}'
        alt=base.with_name(f'{base.stem}_{tag}{base.suffix}')
        if not alt.exists() or same_file(video,alt):
            return alt
        n+=1

def fingerprint(video):
    meta=video.with_suffix(video.suffix+'.h3.json')
    return tuple((f.stat().st_size,f.stat().st_mtime_ns) if f.is_file() else None for f in (video,meta))

def history_files(value):
    if isinstance(value,dict):
        if value.get('type')=='output' and isinstance(value.get('filename'),str):
            yield SOURCE / value.get('subfolder','') / value['filename']
        for child in value.values():
            yield from history_files(child)
    elif isinstance(value,list):
        for child in value:
            yield from history_files(child)

while True:
    candidates=set()
    # Studio writes these sidecars only after final video processing succeeds.
    for meta in SOURCE.rglob('*.h3.json'):
        try:
            json.loads(meta.read_text(encoding='utf-8'))
            candidates.add(meta.with_name(meta.name[:-len('.h3.json')]))
        except (OSError,ValueError):
            pass
    # Also copy completed Comfy outputs if optional Studio metadata was not saved.
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/history?max_items=100',timeout=15) as response:
            history=json.load(response)
        for job in history.values():
            status=job.get('status',{})
            if status.get('completed') and status.get('status_str')=='success':
                candidates.update(history_files(job.get('outputs',{})))
    except Exception as exc:
        print('History check will retry:',type(exc).__name__,str(exc)[:160],flush=True)
    for candidate in sorted(candidates):
        temporary=[]
        try:
            video=candidate.resolve()
            relative=video.relative_to(SOURCE)
            if relative.parts and relative.parts[0]=='h3_enhance_cache':
                continue  # SeedVR2 intermediates; only the final enhanced video is backed up
            if video.suffix.lower() not in VIDEO_EXT or not video.is_file():
                continue
            before=fingerprint(video)
            if not before[0][0] or copied.get(video)==before:
                continue
            pairs=[]
            main_target=drive_target(video)
            for source in (video,video.with_suffix(video.suffix+'.h3.json')):
                if not source.is_file():
                    continue
                target=main_target if source==video else main_target.with_suffix(main_target.suffix+'.h3.json')
                target.parent.mkdir(parents=True,exist_ok=True)
                temp=target.with_name('.'+target.name+'.h3copy-part')
                temporary.append(temp)
                shutil.copy2(source,temp)
                if temp.stat().st_size!=source.stat().st_size:
                    raise OSError('Copy size mismatch')
                pairs.append((temp,target))
            if fingerprint(video)!=before:
                raise OSError('Source changed while copying; retrying')
            for temp,target in pairs:
                os.replace(temp,target)
            copied[video]=before
            note='' if main_target==DEST/relative else f' -> {main_target.name}（Drive 已有同名舊檔，改名保存）'
            print('Drive copy complete:',relative.as_posix()+note,flush=True)
        except Exception as exc:
            print('Drive copy will retry:',candidate,type(exc).__name__,str(exc)[:180],flush=True)
        finally:
            for temp in temporary:
                try:
                    temp.unlink(missing_ok=True)
                except OSError:
                    pass
    time.sleep(5)
"""
copy_path=ROOT/'h3_drive_copy.py'
copy_path.write_text(copy_script,encoding='utf-8')
launch_once('h3_drive_copy',[PYTHON,'-u',copy_path],ROOT,ROOT/'h3_drive_copy.log')
print('影片／圖片備份至:',DRIVE_OUTPUTS_ROOT)
print('請等 /content/h3_drive_copy.log 顯示 Drive copy complete，再刪除執行階段。')


In [ ]:
# 11. 密碼保護的公開網址。先取得 tunnel 域名，再建立嚴格的 Origin 檢查。
launch_once('h3_cloudflared',['/usr/local/bin/cloudflared','tunnel','--no-autoupdate','--protocol','http2','--url','http://127.0.0.1:8081'],ROOT,ROOT/'h3_cloudflared.log')
public_url=None
for _ in range(90):
    text=(ROOT/'h3_cloudflared.log').read_text(errors='replace')
    m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',text)
    if m: public_url=m.group(0); break
    time.sleep(2)
assert public_url, '未取得 tunnel 網址，請查看 h3_cloudflared.log'
# stdin prevents putting the password in process arguments or notebook outputs.
subprocess.run(['htpasswd','-iBc','/content/.h3_basic_auth',BASIC_USER],input=BASIC_PASSWORD+'\n',text=True,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
# Only loopback listeners. Public Host is rewritten for Vite; browser Origin is validated first.
conf=r"""
map $http_origin $h3_origin_ok {
 default 0;
 "" 1;
 "__PUBLIC_URL__" 1;
}
server {
 listen 127.0.0.1:8081;
 server_name _;
 client_max_body_size 2048m;
 auth_basic "H3 Colab Studio";
 auth_basic_user_file /content/.h3_basic_auth;
 proxy_read_timeout 3600s;
 proxy_send_timeout 3600s;
 # These features are explicitly disabled for the Colab edition.
 location = /comfy/h3-studio/output { return 405; }
 location = /comfy/h3-studio/concat { return 405; }
 location = /comfy/h3-studio/continuation-frame { return 405; }
 location = /comfy/h3-studio/optimize-prompt { return 501; }
 location = /comfy/h3-studio/optimize-image-prompt { return 501; }
 location /comfy/h3-studio/drama3/ { return 501; }
 location /comfy/ {
  if ($h3_origin_ok = 0) { return 403; }
  proxy_pass http://127.0.0.1:8188/;
  proxy_http_version 1.1;
  proxy_set_header Host 127.0.0.1:8188;
  proxy_set_header Origin http://127.0.0.1:3000;
  proxy_set_header Upgrade $http_upgrade;
  proxy_set_header Connection "upgrade";
 }
 location / {
  if ($h3_origin_ok = 0) { return 403; }
  proxy_pass http://127.0.0.1:3000;
  proxy_http_version 1.1;
  proxy_set_header Host 127.0.0.1:3000;
  # Dev server rejects module/RSC requests whose Origin != Host (403). Origin already validated above.
  proxy_set_header Origin http://127.0.0.1:3000;
  proxy_set_header X-Forwarded-Proto https;
  proxy_set_header Upgrade $http_upgrade;
  proxy_set_header Connection "upgrade";
 }
}
""".replace('__PUBLIC_URL__',public_url)
Path('/etc/nginx/conf.d/h3.conf').write_text(conf,encoding='utf-8')
# Colab nginx map hostname hash requires larger buckets (verified via nginx -t).
nginx_conf=Path('/etc/nginx/nginx.conf')
nginx_text=nginx_conf.read_text(encoding='utf-8')
if 'map_hash_bucket_size' not in nginx_text:
    assert nginx_text.count('http {') == 1, 'Unexpected nginx.conf format'
    nginx_text=nginx_text.replace('http {','http {\n    map_hash_bucket_size 128;',1)
    nginx_conf.write_text(nginx_text,encoding='utf-8')
default=Path('/etc/nginx/sites-enabled/default')
if default.is_symlink(): default.unlink()
run(['nginx','-t'])
if subprocess.run(['nginx','-s','reload'],capture_output=True).returncode: run(['nginx'])
# reload is async: old workers keep the previous tunnel URL for a moment -> spurious 403.
time.sleep(3)
# Prove the auth/host/origin chain works before displaying the link.
local='http://127.0.0.1:8081'
auth=(BASIC_USER,BASIC_PASSWORD)
assert requests.get(local,timeout=30).status_code==401, '公開代理沒有要求密碼'
for route in ('/','/comfy/system_stats','/comfy/h3-studio/tts/status'):
    r=requests.get(local+route,auth=auth,headers={'Host':public_url.split('://')[1],'Origin':public_url},timeout=60)
    assert r.status_code==200, (route,r.status_code,r.text[:200])
assert requests.get(local+'/comfy/h3-studio/tts/status',auth=auth,headers={'Origin':'https://invalid.example'},timeout=30).status_code==403
for _ in range(20):
    try:
        r=requests.get(public_url,auth=auth,timeout=20)
        if r.status_code==200: break
    except requests.RequestException: pass
    time.sleep(2)
else:
    raise RuntimeError('本機代理已通過，但公開 tunnel 未就緒；請查看日誌，不要提交生成工作')
from IPython.display import display, Markdown
display(Markdown('### [開啟 H3 Colab Studio]('+public_url+')\n帳號：`'+BASIC_USER+'`；密碼為第 1 格設定值。'))
print('首次請先生成短片，確認畫面、聲音與 Drive 備份，再增加片長。')


In [ ]:
# 11b. 生成期間保持 notebook 活躍，並每分鐘回報所有服務狀態（不用另外跑檢查指令）。
# 服務已完全脫離 notebook，按停止這格不會影響 ComfyUI／Studio／tunnel。
import time, json, re, requests
def _alive(name):
    rec=ROOT/(name+'.pid.json')
    try: saved=json.loads(rec.read_text()); return process_token(saved['pid'])==saved['token']
    except Exception: return False
def _ok(url):
    try: return requests.get(url,timeout=10).status_code==200
    except Exception: return False
m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',(ROOT/'h3_cloudflared.log').read_text(errors='replace')) if (ROOT/'h3_cloudflared.log').exists() else None
print('目前 Studio 網址：',m.group(0) if m else '（找不到，請重跑第 11 格或快速恢復）',flush=True)
try:
    while True:
        if not _alive('h3_drive_copy'):
            (ROOT/'h3_drive_copy.pid.json').unlink(missing_ok=True)
            launch_once('h3_drive_copy',[PYTHON,'-u',ROOT/'h3_drive_copy.py'],ROOT,ROOT/'h3_drive_copy.log')
            print('⚠ Drive 備份程式已停止，已自動重啟',flush=True)
        try: q=requests.get('http://127.0.0.1:8188/queue',timeout=10).json(); busy=len(q.get('queue_running',[]))+len(q.get('queue_pending',[]))
        except Exception: busy='?'
        log=ROOT/'h3_drive_copy.log'
        last=log.read_text(errors='replace').strip().splitlines()[-1:] if log.exists() else []
        flag=lambda b:'OK' if b else '✗'
        _bgd=ROOT/'h3_bg_models'
        _bgp=[p.stem for p in _bgd.glob('*.pending')] if _bgd.exists() else []
        _bgf=[p.stem for p in _bgd.glob('*.failed')] if _bgd.exists() else []
        bgs=(' | 背景下載中：'+','.join(_bgp) if _bgp and _alive('h3_bg_models') else (' | ⚠ 背景下載已停止：'+','.join(_bgp) if _bgp else ''))+(' | ⚠ 下載失敗：'+','.join(_bgf)+'（看 h3_bg_models.log）' if _bgf else '')
        snap_log=ROOT/'h3_snapshot.log'
        snap=' | 快照：存檔中' if _alive('h3_snapshot') else (' | 快照：'+snap_log.read_text(errors='replace').strip().splitlines()[-1][-60:] if snap_log.exists() and snap_log.read_text(errors='replace').strip() else '')
        print(time.strftime('%H:%M'),f"ComfyUI {flag(_ok('http://127.0.0.1:8188/system_stats'))}",f"Studio {flag(_ok('http://127.0.0.1:3000'))}",
              f"tunnel {flag(_alive('h3_cloudflared'))}",'佇列:',busy,'|',(last[0][-80:] if last else '尚無備份紀錄')+snap+bgs,flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    print('已停止保持活躍（服務仍在背景執行）')


In [ ]:
# R. 快速恢復（kernel 重啟／當機後用）：不重裝、不重 build，直接重啟所有服務並給新網址。
# 前提：這個執行階段之前已完整跑過第 1～11 格。
from google.colab import drive
from pathlib import Path
import os, sys, json, signal, time, re, subprocess, getpass
drive.mount('/content/drive')
ROOT=Path('/content'); DRIVE_ROOT=ROOT/'drive/MyDrive/H3_Colab'; OUTPUTS_ROOT=ROOT/'h3_outputs'; DRIVE_OUTPUTS_ROOT=DRIVE_ROOT/'outputs'
REPO=ROOT/'LocalStudio'; COMFY=ROOT/'ComfyUI'; WEB=REPO/'h3-local-studio'; VENV=ROOT/'h3_venv'; PYTHON=VENV/'bin/python'
BASIC_USER='h3'
while True:
    BASIC_PASSWORD=getpass.getpass('Studio 密碼（至少 8 字元；登入帳號是 h3）：')
    if len(BASIC_PASSWORD)>=8: break
    print(f'⚠ 密碼只有 {len(BASIC_PASSWORD)} 個字元，至少要 8 個，請重新輸入。',flush=True)
GEMINI_KEY=getpass.getpass('Gemini TTS API key（選填，Enter 略過）：').strip()
import requests, requests.utils
requests.utils.DEFAULT_ACCEPT_ENCODING='gzip, deflate'
def run(args,**kw): subprocess.run([str(x) for x in args],check=True,**kw)
def process_token(pid):
    try:
        fields=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
        return None if fields[0]=='Z' else fields[19]
    except (FileNotFoundError,ProcessLookupError): return None
def stop_managed(name):
    record=ROOT/(name+'.pid.json')
    if not record.exists(): return
    saved=json.loads(record.read_text()); pid=saved['pid']
    if process_token(pid)==saved['token']:
        os.killpg(pid,signal.SIGTERM)
        for _ in range(50):
            if process_token(pid)!=saved['token']: break
            time.sleep(0.1)
        if process_token(pid)==saved['token']: os.killpg(pid,signal.SIGKILL)
    record.unlink(missing_ok=True)
DETACH="import subprocess,sys,json,os;p=subprocess.Popen(json.loads(sys.argv[1]),cwd=sys.argv[2],stdout=open(sys.argv[3],'w'),stderr=subprocess.STDOUT,stdin=subprocess.DEVNULL,start_new_session=True);print(p.pid,flush=True);os._exit(0)"
def launch_once(name,command,cwd,log,env=None):
    record=ROOT/(name+'.pid.json')
    if record.exists():
        saved=json.loads(record.read_text())
        if process_token(saved['pid'])==saved['token']: raise RuntimeError(name+' 已啟動')
        record.unlink()
    out=subprocess.run([sys.executable,'-c',DETACH,json.dumps([str(x) for x in command]),str(cwd),str(log)],capture_output=True,text=True,env=env,check=True,start_new_session=True)
    pid=int(out.stdout.strip()); time.sleep(0.3)
    token=process_token(pid); assert token, name+' 立即退出，請看 '+str(log)
    record.write_text(json.dumps({'pid':pid,'token':token}),encoding='utf-8'); print('Started',name,pid)
def wait_url(url,seconds=300):
    until=time.monotonic()+seconds
    while time.monotonic()<until:
        try:
            r=requests.get(url,timeout=60)
            if r.status_code==200: return r
        except requests.RequestException: pass
        time.sleep(2)
    raise RuntimeError('服務未就緒：'+url)
# Keep the previous logs (快速恢復 overwrites them) for post-mortem.
import shutil
for n in ('h3_studio','h3_comfy','h3_drive_copy','h3_cloudflared'):
    if (ROOT/(n+'.log')).exists(): shutil.copy(ROOT/(n+'.log'),ROOT/(n+'.log.prev'))
for name in ('h3_cloudflared','h3_drive_copy','h3_studio','h3_comfy'): stop_managed(name)
# mk29：接續未完成的 FlashVSR／VOSR 背景下載（下載可續傳）。
if any((ROOT/'h3_bg_models').glob('*.pending')) and (ROOT/'h3_bg_models.py').is_file():
    stop_managed('h3_bg_models')
    launch_once('h3_bg_models',[PYTHON,'-u',ROOT/'h3_bg_models.py',ROOT/'h3_bg_models.json'],ROOT,ROOT/'h3_bg_models.log')
assert (WEB/'dist').is_dir() and (ROOT/'h3_drive_copy.py').is_file(), '這個執行階段沒有完整跑過第 1～11 格，請從頭執行'
service_env=os.environ.copy(); service_env['GEMINI_TTS_API_KEY']=GEMINI_KEY
launch_once('h3_comfy',[PYTHON,'main.py','--listen','127.0.0.1','--port','8188','--enable-cors-header','--disable-auto-launch','--extra-model-paths-config','extra_model_paths.yaml','--output-directory',OUTPUTS_ROOT],COMFY,ROOT/'h3_comfy.log',service_env)
wait_url('http://127.0.0.1:8188/system_stats')
launch_once('h3_studio',['npm','run','start','--','--hostname','127.0.0.1','--port','3000'],WEB,ROOT/'h3_studio.log')
wait_url('http://127.0.0.1:3000')
launch_once('h3_drive_copy',[PYTHON,'-u',ROOT/'h3_drive_copy.py'],ROOT,ROOT/'h3_drive_copy.log')
launch_once('h3_cloudflared',['/usr/local/bin/cloudflared','tunnel','--no-autoupdate','--protocol','http2','--url','http://127.0.0.1:8081'],ROOT,ROOT/'h3_cloudflared.log')
public_url=None
for _ in range(90):
    m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',(ROOT/'h3_cloudflared.log').read_text(errors='replace'))
    if m: public_url=m.group(0); break
    time.sleep(2)
assert public_url, '未取得 tunnel 網址'
subprocess.run(['htpasswd','-iBc','/content/.h3_basic_auth',BASIC_USER],input=BASIC_PASSWORD+'\n',text=True,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
conf=Path('/etc/nginx/conf.d/h3.conf'); text=conf.read_text(encoding='utf-8')
text=re.sub(r'"https://[a-zA-Z0-9-]+\.trycloudflare\.com" 1;','"'+public_url+'" 1;',text); conf.write_text(text,encoding='utf-8')
run(['nginx','-t'])
if subprocess.run(['nginx','-s','reload'],capture_output=True).returncode: run(['nginx'])
time.sleep(3)
local='http://127.0.0.1:8081'; auth=(BASIC_USER,BASIC_PASSWORD); host=public_url.split('://')[1]
for _ in range(15):
    if requests.get(local+'/',auth=auth,headers={'Host':host,'Origin':public_url},timeout=60).status_code==200: break
    time.sleep(2)
else: raise RuntimeError('本機代理檢查失敗')
for _ in range(20):
    try:
        if requests.get(public_url,auth=auth,timeout=20).status_code==200: break
    except requests.RequestException: pass
    time.sleep(2)
from IPython.display import display, Markdown
display(Markdown('### [開啟 H3 Colab Studio]('+public_url+')\n帳號：`'+BASIC_USER+'`'))
print('已恢復。接著執行 11b 保持活躍。')


In [ ]:
# 12. 查看日誌與尚未備份的已完成影片／圖片（可重複執行，不送 GPU 工作）。
for name in ('h3_comfy','h3_studio','h3_cloudflared','h3_drive_copy','h3_snapshot','h3_bg_models'):
    log=ROOT/(name+'.log')
    print('\n===',name,'===')
    print(log.read_text(errors='replace')[-4000:] if log.exists() else 'not started')
pending=[]
for meta in OUTPUTS_ROOT.rglob('*.h3.json'):
    video=meta.with_name(meta.name[:-len('.h3.json')])
    if video.suffix.lower() not in {'.mp4','.webm','.mov','.png','.jpg','.jpeg','.webp'} or not video.is_file(): continue
    base=DRIVE_OUTPUTS_ROOT/video.relative_to(OUTPUTS_ROOT)
    def _done(target):
        target_meta=target.with_suffix(target.suffix+'.h3.json')
        return target.is_file() and target.stat().st_size==video.stat().st_size and target_meta.is_file() and target_meta.read_bytes()==meta.read_bytes()
    # Renamed copies (name_YYYYMMDD-HHMMSS.ext) count too.
    if not (_done(base) or any(_done(x) for x in base.parent.glob(base.stem+'_*'+base.suffix))):
        pending.append(str(video.relative_to(OUTPUTS_ROOT)))
print('尚未確認備份完成:',pending if pending else '沒有（仍請查看 copy 日誌）')


## 停止服務（需要停止時才執行）
停止前先執行上一格，等所有影片顯示 `Drive copy complete`，確認 Drive 中能開啟。停止服務不等於刪除 Colab 執行階段，也不代表停止 CU 計費；不使用時請在 Colab 選「中斷連線並刪除執行階段」。


In [ ]:
for name in ('h3_bg_models','h3_cloudflared','h3_drive_copy','h3_studio','h3_comfy'):
    stop_managed(name)
print('已停止 notebook 管理的服務；Drive 備份不會刪除。')
